# Notebook 1 Time Series Jena Climate - GRU Model

### Developed by:

- António Cruz (140129)
- Cátia Brás (120093)
- Ricardo Kyaseller (95813)

# 1. Introduction

Time series forecasting plays a central role in meteorology, environmental monitoring, and climate-aware decision-making. The availability of long-term, high-resolution atmospheric measurements enables the development of deep learning models capable of capturing complex temporal dependencies and nonlinear relationships in environmental systems.

This project focuses on the Jena Climate dataset, a multivariate meteorological time series collected between 2009 and 2016 at the Max Planck Institute for Biogeochemistry. The dataset contains 14 atmospheric variables recorded every 10 minutes, including temperature, pressure, humidity, and wind-related measurements. 

However, in accordance with the project guidelines, only a subset of variables is used in the experiments in order to avoid near-deterministic relationships with the target variable. The input variables considered in this study are: past air temperature (T (degC)), atmospheric pressure (p (mbar)), relative humidity (rh (%)), wind speed (wv (m/s)), maximum wind speed (max. wv (m/s)), and wind direction (wd (deg)), together with the corresponding timestamp.

Following the project guidelines, the forecasting task is formulated as a multivariate-input, univariate-output, multi-step forecasting problem, where past meteorological observations are used to predict the future air temperature over a fixed horizon of 24 hours. The input variables include past air temperature together with atmospheric pressure, relative humidity, wind speed, maximum wind speed, and wind direction.

To address this problem, we implement and evaluate deep learning models designed for sequential data, namely Gated Recurrent Units (GRU) and Transformer-based architectures. The study follows a complete time series modelling pipeline including exploratory data analysis, data preprocessing, temporal resampling, windowing strategies, model development, hyperparameter exploration, and performance evaluation.

The final objective is not only to obtain accurate forecasts, but also to understand how different architectures and modelling choices influence forecasting performance, particularly across different forecast horizons.

# 2. Methodology

## 2.1 Computational Environment

The experiments were conducted using Python and a set of widely adopted scientific computing libraries. Data manipulation and preprocessing were performed using pandas and numpy, while matplotlib was used for exploratory visualizations. The deep learning models were implemented with TensorFlow/Keras, which provides efficient tools for constructing and training recurrent neural networks such as GRUs.

Additional utilities included scikit‑learn for normalization and evaluation metrics (MAE, RMSE, R²), Optuna for hyperparameter optimization, and standard Python modules such as random and time for reproducibility and runtime management.

This computational setup ensures a robust and reproducible environment for all stages of the forecasting pipeline.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import random
import time
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
print(tf.config.experimental.get_device_details(gpus[0]))

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)


## 2.2 Dataset Description

The dataset used in this project is the Jena Climate dataset, recorded by the [Max Planck Institute for Biogeochemistry](https://www.bgc-jena.mpg.de/wetter/). It contains 14 meteorological variables measured every 10 minutes over the period January 10 2009 to December 31 2016. According to the dataset documentation, “The dataset consists of 14 features recorded once per 10 minutes.

The measurements include atmospheric pressure, temperature, humidity, wind characteristics, and several derived thermodynamic variables. The table below summarises the available features, their formats, and their physical meaning.

The table below shows the column names, their value formats, and their description.


Index| Features      |Format             |Description
-----|---------------|-------------------|-----------------------
1    |Date Time      |01.01.2009 00:10:00|Date-time reference
2    |p (mbar)       |996.52             |The pascal SI derived unit of pressure used to quantify internal pressure. Meteorological reports typically state atmospheric pressure in millibars.
3    |T (degC)       |-8.02              |Temperature in Celsius
4    |Tpot (K)       |265.4              |Temperature in Kelvin
5    |Tdew (degC)    |-8.9               |Temperature in Celsius relative to humidity. Dew Point is a measure of the absolute amount of water in the air, the DP is the temperature at which the air cannot hold all the moisture in it and water condenses.
6    |rh (%)         |93.3               |Relative Humidity is a measure of how saturated the air is with water vapor, the %RH determines the amount of water contained within collection objects.
7    |VPmax (mbar)   |3.33               |Saturation vapor pressure
8    |VPact (mbar)   |3.11               |Vapor pressure
9    |VPdef (mbar)   |0.22               |Vapor pressure deficit
10   |sh (g/kg)      |1.94               |Specific humidity
11   |H2OC (mmol/mol)|3.12               |Water vapor concentration
12   |rho (g/m ** 3) |1307.75            |Airtight
13   |wv (m/s)       |1.03               |Wind speed
14   |max. wv (m/s)  |1.75               |Maximum wind speed
15   |wd (deg)       |152.3              |Wind direction in degrees

**Table 1** - Jena Climate Dataset: Feature Overview

## 2.3 Data Acquisition and Initial Cleaning

The Jena Climate dataset was retrieved from the official TensorFlow/Keras repository, which provides a compressed CSV file containing the full time series. The data were downloaded and extracted programmatically using Python’s `zipfile` module and Keras utilities.

After loading the dataset into a pandas DataFrame, several preprocessing steps were applied:

* Sensor error correction: Wind speed columns (`wv (m/s)` and `max. wv (m/s)`) contained erroneous values of `-9999.0`, corresponding to equipment failures. These values were replaced with `NaN` to ensure they would be ignored during resampling and aggregation.
* Datetime parsing: The `Date Time` column was originally stored as a string in European format (`DD.MM.YYYY HH:MM:SS`). It was converted to a proper `datetime` object using `pd.to_datetime()` with an explicit format string. This ensures correct temporal indexing and compatibility with time-based operations.
* Initial inspection: The first few rows of the dataset were examined using `df.head(5)` to confirm successful loading and formatting. The output confirmed the presence of all expected variables, with realistic values and timestamps starting from January 1st, 2009.

These steps ensure that the dataset is clean, correctly formatted, and ready for further preprocessing and analysis.

In [2]:
"""
from zipfile import ZipFile

uri = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
zip_path = keras.utils.get_file(origin=uri, fname="jena_climate_2009_2016.csv.zip")
zip_file = ZipFile(zip_path)
zip_file.extractall()
"""

csv_path = "data/jena_climate_2009_2016.csv"
df = pd.read_csv(csv_path)

# Fix sensor error codes: -9999.0 in wind speed columns are equipment failures
# (18 rows in wv, 20 rows in max.wv — all from 2015-07-13 09:00–12:00).
# Replace with NaN so hourly .mean() resampling ignores them.
df[["wv (m/s)", "max. wv (m/s)"]] = df[["wv (m/s)", "max. wv (m/s)"]].replace(-9999.0, np.nan)

In [3]:
# Convert string to a proper DateTime so that
# matplotlib and the other components won't have to do it on every time
# Also, the date format in the CSV is European: DD.MM.YYYY HH:MM:SS (day comes first)
# Fix that by specifying the format explicitly:
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")

In [4]:
df.head(5)

## 2.5 Exploratory Data Analysis (EDA)

The exploratory data analysis aims to assess the structure, quality, and statistical behaviour of the Jena Climate dataset before applying any modelling techniques. This includes inspecting dataset dimensions, identifying missing or duplicated values, analysing variable distributions, detecting outliers, and examining correlations between meteorological variables.


### 2.5.1 Dataset Structure

The dataset contains 420,551 observations and 14 meteorological variables, together with a timestamp column (`Date Time`). As confirmed by the output of `df.shape`, the `Date Time` column is stored as a `datetime64` object, while all meteorological variables are numerical (`float64`). 

The timestamp is used as the temporal reference for the series and allows operations such as indexing, resampling, and windowing. This structure is well suited for time-series modelling and facilitates the application of sequential deep learning models.

In [5]:
df.shape , df.dtypes
print('\nDataset Shape is:', df.shape)
print('\nDataset Types are:', df.dtypes)

### 2.5.2 Missing Values

An inspection of missing values using `df.isnull().sum()` shows that the dataset is almost entirely complete.  
Only two variables contain missing entries:

* wv (m/s): 18 missing values  
* max. wv (m/s): 20 missing values  

These missing values correspond to sensor failures originally encoded as `-9999.0`, which were intentionally replaced with `NaN` during the cleaning process.  
No other variables contain missing data.


In [6]:
df.isnull().sum() 

### 2.5.3 Duplicate Records

A total of 327 duplicated rows were identified using `df.duplicated().sum()`. Since each duplicated record appears twice when both the original and repeated entries are considered, this corresponds to 654 duplicated observations in total. Inspection of a sample of these records showed that the duplicated rows were exact copies, including identical timestamps and meteorological values.

These duplicates are likely due to repeated logging or data collection redundancies rather than meaningful repeated measurements. To ensure temporal consistency and avoid bias in the statistical analysis, resampling step, and model training, duplicated rows were removed using `drop_duplicates(keep="first")`.

After this operation, no duplicated rows remained in the dataset, and the final shape was reduced from 420,551 to 420,224 observations.

In [7]:
df.duplicated().sum()

In [8]:
dup_sample = (
    df[df.duplicated(keep=False)]
    .sort_values(list(df.columns))
)

print("Total duplicated rows (including originals):", len(dup_sample))
display(dup_sample.head(20))

### 2.5.3.1 Drop Duplicated Values

In [9]:
df = df.drop_duplicates(keep="first")

In [10]:
print("Remaining duplicates:", df.duplicated().sum())
print("New shape:", df.shape)

### 2.5.4 Descriptive Statistics

Descriptive statistics were computed for all numerical variables using `df.describe().T` in order to obtain a first overview of the scale and variability of the meteorological measurements.

Key observations include:

* Temperature (T) ranges from –23.01°C to 37.28°C, reflecting substantial seasonal variability over the observation period.
* Relative humidity (rh) spans from 12.95% to 100%, indicating a wide range of atmospheric moisture conditions.
* Wind speed (wv) ranges from 0 to 28.49 m/s, suggesting generally low wind conditions with occasional high-speed events.
* Atmospheric pressure (p) varies between 913.60 mbar and 1015.35 mbar, which is consistent with typical pressure ranges observed in mid-latitude weather systems.

These statistics provide an initial understanding of the scale and variability of the meteorological variables and help guide subsequent preprocessing and modelling decisions.

Additionally, the statistics confirm that all variables fall within physically plausible ranges, indicating that no major data corruption or unrealistic values are present in the dataset.

In [11]:
df.describe(include=[np.number]).T

### 2.5.5 Time-Series Visualization

To better understand the temporal behaviour of the meteorological variables, raw time‑series plots were generated for all features in the dataset. These visualizations allow the identification of long‑term trends, seasonal patterns, and abrupt changes in atmospheric conditions.

The plots reveal clear annual seasonality in temperature‑related variables (T, Tpot, Tdew), with warm summers and cold winters repeating consistently across the years. Humidity‑related variables (rh, VPact, sh, H2OC) also exhibit seasonal fluctuations, typically increasing during colder months.

Wind‑related variables (wv, max. wv, wd) show high short‑term variability, reflecting the turbulent nature of wind dynamics. Atmospheric pressure (p) displays slower oscillations associated with synoptic‑scale weather systems.

These temporal patterns confirm that the dataset contains rich and diverse dynamics, which must be captured by the forecasting models.

In [12]:
titles = [
    "Pressure",
    "Temperature",
    "Temperature in Kelvin",
    "Temperature (dew point)",
    "Relative Humidity",
    "Saturation vapor pressure",
    "Vapor pressure",
    "Vapor pressure deficit",
    "Specific humidity",
    "Water vapor concentration",
    "Airtight",
    "Wind speed",
    "Maximum wind speed",
    "Wind direction in degrees",
]

feature_keys = [
    "p (mbar)",
    "T (degC)",
    "Tpot (K)",
    "Tdew (degC)",
    "rh (%)",
    "VPmax (mbar)",
    "VPact (mbar)",
    "VPdef (mbar)",
    "sh (g/kg)",
    "H2OC (mmol/mol)",
    "rho (g/m**3)",
    "wv (m/s)",
    "max. wv (m/s)",
    "wd (deg)",
]

colors = [
    "blue",
    "orange",
    "green",
    "red",
    "purple",
    "brown",
    "pink",
    "gray",
    "olive",
    "cyan",
]

date_time_key = "Date Time"


def show_raw_visualization(data):
    time_data = data[date_time_key]
    fig, axes = plt.subplots(
        nrows=7, ncols=2, figsize=(15, 20), dpi=80, facecolor="w", edgecolor="k"
    )
    for i in range(len(feature_keys)):
        key = feature_keys[i]
        c = colors[i % (len(colors))]
        t_data = data[key]
        t_data.index = time_data
        t_data.head()
        ax = t_data.plot(
            ax=axes[i // 2, i % 2],
            color=c,
            title="{} - {}".format(titles[i], key),
            rot=25,
        )
        ax.legend([titles[i]])
    plt.tight_layout()


show_raw_visualization(df)


**Figure 1** - Time‑Series Overview of the Jena Climate Variables

### 2.5.6 Identification of Numerical Variables

Before analysing variable distributions, all numerical features in the dataset were identified using `df.select_dtypes(include=[np.number])`. A total of 14 numerical variables were detected, corresponding to the meteorological measurements recorded by the weather station.

These variables include temperature‑related features (T, Tpot, Tdew), humidity‑related features (rh, VPact, sh, H2OC), pressure (p), wind‑related variables (wv, max. wv, wd), and other atmospheric indicators. Identifying the numerical columns ensures that subsequent statistical and distributional analyses are applied consistently across all quantitative features.

In [13]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", num_cols)
print("Total numeric:", len(num_cols))

### 2.5.7 Distribution Analysis (Histograms)

To analyse the empirical distribution of the meteorological variables, histograms were generated for all numerical features in the dataset. This step helps identify skewness, concentration of values, and potential extreme observations that may influence forecasting models.

The histograms reveal that many variables exhibit non-Gaussian distributions:

* Wind-related variables (wv, max. wv) show strong right-skewness due to occasional high-intensity wind events.
* Humidity-related variables (rh, VPact, sh, H2OC) tend to cluster near their physical limits, reflecting atmospheric saturation behaviour.
* Temperature-related variables (T, Tpot, Tdew) exhibit broad distributions reflecting seasonal variability over the observation period.
* Atmospheric pressure (p) shows a narrow distribution with relatively low variance compared with other variables.

These observations highlight the importance of applying normalization before model training and motivate the use of deep learning models capable of capturing non-linear relationships in meteorological time series.

In [14]:
for c in num_cols:
    plt.figure()
    plt.hist(df[c].dropna(), bins=50)
    plt.title(f"Histogram - {c}")
    plt.xlabel(c)
    plt.ylabel("Count")
    plt.show()

### 2.5.8 Boxplots

Boxplots were generated for all numerical variables to visualise their dispersion, central tendency, and potential extreme observations. These plots provide a compact summary of the median, interquartile range (IQR), and values outside the IQR range.

Most variables exhibit a considerable number of statistical outliers according to the IQR criterion. In large meteorological datasets, such observations often correspond to rare but physically plausible weather events rather than measurement errors.

Wind-related variables (wv and max. wv) and vapor pressure deficit (VPdef) display particularly large ranges and numerous extreme observations, reflecting the intermittent nature of wind events and atmospheric moisture dynamics. In contrast, atmospheric pressure (p) shows a relatively narrow distribution with limited variability.

Overall, the boxplots confirm the presence of natural variability and occasional extreme values in the dataset, which is typical for environmental measurements.

In [15]:
for c in num_cols:
    plt.figure()
    plt.boxplot(df[c].dropna(), vert=False)
    plt.title(f"Boxplot - {c}")
    plt.xlabel(c)
    plt.show()

### 2.5.9 Outlier Detection

Outliers were identified using the Interquartile Range (IQR) method. For each numerical variable, observations below \(Q1 - 1.5 \times IQR\) or above \(Q3 + 1.5 \times IQR\) were flagged as statistical outliers.

The variables with the highest number of detected outliers include:

* VPdef (mbar): approximately 32,000 observations  
* wv (m/s) and max. wv (m/s): together more than 26,000 observations  
* VPmax (mbar): over 11,000 observations  
* p (mbar): approximately 6,700 observations  

It is important to note that the IQR method identifies *statistical* outliers rather than erroneous measurements. In meteorological datasets, such values often correspond to rare but physically plausible atmospheric conditions, including strong wind events or extreme humidity variations.

Given the large size of the dataset and the fact that these observations fall within physically realistic ranges, no outlier removal was performed. Retaining these values preserves the natural variability of the atmospheric processes represented in the dataset.

In [16]:
outlier_summary = []

for c in num_cols:
    s = df[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((s < lower) | (s > upper)).sum()
    outlier_summary.append([c, n_outliers, lower, upper])

outliers_df = pd.DataFrame(outlier_summary, columns=["col", "n_outliers", "iqr_low", "iqr_high"])
display(outliers_df.sort_values("n_outliers", ascending=False))

### 2.5.10 Correlation Between Variables

A correlation matrix was computed to analyse linear relationships between the numerical variables. The resulting heatmap reveals several strong dependencies between atmospheric variables.

Temperature-related variables (T, Tpot, Tdew) show strong positive correlations with vapor pressure and humidity-related variables such as VPmax, VPact, sh, and H2OC. These relationships are expected, since atmospheric moisture capacity is strongly influenced by temperature.

Air density (rho) is strongly negatively correlated with temperature (≈ −0.96), which is consistent with the physical relationship described by the ideal gas law.

Wind-related variables (wv, max. wv, wd) exhibit relatively weak correlations with most other variables, indicating that wind dynamics behave more independently from temperature and humidity conditions.

The correlation matrix also reveals extremely strong correlations among several humidity-related variables, particularly between sh, H2OC, and VPact, where correlations approach values close to 1. These near-deterministic relationships indicate a high degree of redundancy between certain variables.

Such redundancy between variables may introduce multicollinearity, which can negatively affect model interpretability and training stability. For this reason, and following the project guidelines, only a subset of meteorological variables is used during modelling to avoid introducing redundant predictors.

In [17]:
corr = df[num_cols].corr()

display(corr)

plt.figure(figsize=(8, 6))
plt.imshow(corr.values)
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title("Correlation Matrix")
plt.colorbar()
plt.tight_layout()
plt.show()

**Figure 30** - Correlation Matrix of Meteorological Variables


### 2.5.11 Correlation with Target Variable (Temperature)

To better understand which variables are most informative for forecasting temperature, the correlation of each feature with the target variable T (degC) was computed.

The strongest positive correlations include:

* Tpot (K)
* VPmax (mbar)
* Tdew (degC)
* VPact (mbar)
* sh (g/kg)
* H2OC (mmol/mol)

These strong positive correlations reflect the physical relationships between temperature, atmospheric moisture capacity, and vapor pressure dynamics.

Strong negative correlations include:

* rho (g/m³)
* rh (%)

In particular, air density (rho) shows a very strong negative relationship with temperature (≈ −0.96), which is consistent with the ideal gas law.

Wind-related variables (wv, max. wv, wd) exhibit weak correlations with temperature, suggesting they provide limited direct linear predictive power for this target variable. However, they may still contribute indirectly through nonlinear relationships captured by deep learning models.

It is also important to note that several highly correlated variables will not be used during model training in order to avoid redundancy, following the project guidelines.

| Variable | Correlation with T (degC) |
|--------|---------------------------|
| T (degC) | 1.000 |
| Tpot (K) | 0.997 |
| VPmax (mbar) | 0.951 |
| Tdew (degC) | 0.896 |
| VPact (mbar) | 0.868 |
| H2OC (mmol/mol) | 0.867 |
| sh (g/kg) | 0.867 |
| VPdef (mbar) | 0.762 |
| max. wv (m/s) | 0.125 |
| wv (m/s) | 0.088 |
| wd (deg) | 0.039 |
| p (mbar) | -0.045 |
| rh (%) | -0.572 |
| rho (g/m³) | -0.963 |

**Table 2** - Correlation of All Variables with Temperature (T (°C))

In [18]:
target = "T (degC)"
corr_target = corr[target].sort_values(ascending=False)

display(corr_target)

plt.figure()
plt.bar(corr_target.index, corr_target.values)
plt.xticks(rotation=90)
plt.title("Correlation with Temperature")
plt.ylabel("Correlation")
plt.tight_layout()
plt.show()

**Figure 31** - Correlation with Temperature (Bar Chart)

### 2.5.12 Temporal Evolution of Temperature

A time-series plot of air temperature over the observation period (2009–2016) reveals clear seasonal patterns. The data exhibits strong annual cycles, with consistently warmer temperatures during summer months and colder temperatures during winter periods.

In addition to these long-term seasonal patterns, the series also shows substantial short-term variability, reflecting daily fluctuations and synoptic-scale weather dynamics.

The absence of a strong long-term trend suggests that the temperature dynamics are primarily driven by seasonal and short-term atmospheric processes rather than structural changes over time.

These observations confirm that the dataset contains pronounced temporal structure and periodic components, which must be captured by the forecasting models. Sequential deep learning architectures such as recurrent neural networks and transformers are particularly well suited for modelling such temporal dependencies.

In [19]:
plt.figure(figsize=(10, 4))
plt.plot(df["Date Time"], df["T (degC)"])
plt.title("Temperature over time")
plt.xlabel("Date")
plt.ylabel("T (degC)")
plt.show()

**Figure 32** - Temperature Time Series (2009-2017)

### 2.5.13 Mean Temperature by Hour of Day

To analyse diurnal patterns, the average temperature was computed for each hour of the day. The resulting profile reveals a clear daily cycle in temperature.

Minimum temperatures occur during the early morning hours, followed by a steady increase throughout the morning. Peak temperatures are observed in the mid-afternoon, after which temperatures gradually decline during the evening and nighttime hours.

This pattern confirms the presence of strong daily periodicity in the dataset. Such cyclic behaviour is particularly relevant for short-term forecasting and motivates the inclusion of time-based features (e.g., hour-of-day encodings) to help the model capture daily temperature dynamics.

In [20]:
# Strip any accidental whitespace from column names (defensive, no-op if already clean)
df.columns = df.columns.str.strip()

In [21]:
df["hour"] = df["Date Time"].dt.hour
hourly_mean = df.groupby("hour")["T (degC)"].mean()

plt.figure()
plt.bar(hourly_mean.index, hourly_mean.values)
plt.title("Average Temperature by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Mean T (degC)")
plt.show()

**Figure 33** - Average Temperature by Hour of Day

### 2.5.14 Annual Seasonality

To analyse long‑term seasonal behaviour, the average temperature was computed for each month of the year. The results show a clear annual cycle, with minimum temperatures occurring in winter (January and December) and maximum temperatures during summer (July and August). This confirms the presence of strong yearly seasonality in the dataset, which is relevant for medium‑ and long‑term forecasting.


In [22]:
df["month"] = df["Date Time"].dt.month
monthly_mean = df.groupby("month")["T (degC)"].mean()

plt.figure()
plt.bar(monthly_mean.index, monthly_mean.values)
plt.title("Average Temperature by Month")
plt.xlabel("Month")
plt.ylabel("Mean T (degC)")
plt.show()

**Figure 34** - Average Temperature by Month


### 2.5.15 Relationship Between the Target Variable and the Remaining Features

Scatter plots were generated to visualise the relationship between the target variable, T (degC), and the remaining numerical features. These plots help assess the strength, direction, and functional form of the dependencies.

Very strong relationships were observed for variables such as Tpot (K), Tdew (degC), VPmax (mbar), and rho (g/m³). In particular, Tpot (K) shows an almost perfectly linear positive relationship with temperature, while rho (g/m³) exhibits a very strong negative relationship, consistent with the physical behaviour of air density.

Other variables, such as VPmax (mbar), VPact (mbar), sh (g/kg), and H2OC (mmol/mol), display strong but clearly non-linear relationships with temperature. These patterns reflect the physical dependence between temperature and atmospheric moisture-related quantities.

In contrast, wind-related variables (wv, max. wv, and wd) show much more diffuse scatter patterns, suggesting weaker direct linear relationships with the target variable.

Overall, these plots indicate that temperature is strongly related to several atmospheric variables, often through non-linear dependencies. This supports the use of deep learning models capable of capturing complex relationships beyond simple linear effects.

Although all variables were analysed during exploratory data analysis, only the subset specified in the project guidelines was retained for modelling.

In [23]:
target = "T (degC)"
exclude = {target, "hour", "month"}
x_cols = [c for c in num_cols if c not in exclude]

for x in x_cols:
    plt.figure()
    plt.scatter(df[x], df[target], s=1, alpha=0.3)
    plt.xlabel(x)
    plt.ylabel(target)
    plt.title(f"{target} vs {x}")
    plt.show()

### 2.5.16 Target Mean Across Binned Variables

To further analyse the functional relationship between temperature and the remaining variables, each feature was divided into quantile-based bins and the mean temperature was computed within each bin. This approach helps reveal the underlying trend between variables while reducing the effect of noise present in raw scatter plots.

Several clear patterns emerge:

* Tpot (K) and Tdew (degC) show almost perfectly linear relationships with temperature.
* VPmax (mbar), VPact (mbar), sh (g/kg), and H2OC (mmol/mol) exhibit strong but clearly non-linear increasing relationships with temperature.
* rho (g/m³) shows a strong monotonic negative relationship with temperature, consistent with the physical relationship between temperature and air density.
* Relative humidity (rh) displays a decreasing relationship with temperature.
* Wind-related variables (wv, max. wv, wd) show relatively weak and noisy patterns, suggesting limited direct predictive power.

These results confirm that temperature is strongly related to several atmospheric variables, often through non-linear relationships. This further motivates the use of deep learning models capable of capturing complex non-linear dependencies in time series forecasting.

In [24]:
target = "T (degC)"
exclude = {target, "hour", "month"}
x_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]

df_s = df.sample(50000, random_state=42)

for x in x_cols:
    # dividir x em 30 intervalos e ver média de T em cada intervalo
    bins = pd.cut(df_s[x], bins=30)
    m = df_s.groupby(bins)[target].mean()

    plt.figure()
    plt.plot(range(len(m)), m.values)
    plt.title(f"Mean {target} across {x} bins")
    plt.xlabel("Bin index (low -> high)")
    plt.ylabel(f"Mean {target}")
    plt.show()

### 2.5.17 Variables Mean by Hour of Day

The mean value of selected variables (temperature, relative humidity, pressure and wind speed) was computed for each hour of the day to characterise diurnal patterns.

Temperature follows a typical daily cycle, with minimum values in the early morning and maximum values in the mid‑afternoon. Relative humidity shows the opposite behaviour, peaking around dawn and decreasing throughout the day. Pressure and wind speed also exhibit systematic daily variations, although with smaller amplitude. These results confirm the presence of strong intra‑day seasonality in the dataset.


In [25]:
df["hour"] = df["Date Time"].dt.hour

vars_to_check = ["T (degC)", "rh (%)", "p (mbar)", "wv (m/s)"]
for v in vars_to_check:
    m = df.groupby("hour")[v].mean()
    plt.figure()
    plt.plot(m.index, m.values)
    plt.title(f"Mean {v} by hour")
    plt.xlabel("Hour")
    plt.ylabel(v)
    plt.show()

# 3. Data Understanding – Detailed Analysis

This section provides a deeper examination of the temporal structure and statistical characteristics of the dataset, focusing on aspects that are particularly relevant for time-series forecasting.

The dataset is first sorted chronologically using the `Date Time` column to ensure the correct temporal ordering of observations. The forecasting target is defined as air temperature in degrees Celsius, `T (degC)`, while the remaining meteorological variables are considered potential predictors.

The following subsections analyse key properties of the time series, including temporal regularity, seasonality patterns, resampling behaviour, and the presence of missing values. Understanding these aspects is essential for designing appropriate preprocessing steps and for selecting suitable modelling approaches for multivariate time-series forecasting.


In [26]:
df = df.sort_values("Date Time").reset_index(drop=True)

target = "T (degC)"

## 3.1 Temporal Regularity and Gaps

To assess the temporal consistency of the dataset, the time differences between consecutive observations were computed. The most common interval between measurements is 10 minutes, confirming that the dataset follows a regular 10-minute sampling frequency.

A small number of irregular intervals were detected. In total, only five gaps larger than the expected sampling interval were observed. These include short gaps of 20–30 minutes as well as two larger gaps (16 hours and approximately 3 days), which likely correspond to temporary sensor outages or data logging interruptions.

The small number of irregular intervals indicates that the dataset is highly consistent from a temporal perspective. Nevertheless, identifying these gaps is important before performing resampling operations or applying time-series models that assume uniform temporal spacing.

| Time Interval (Delta) | Count |
|----------------------|------|
| 00:10:00             | 420218 |
| 00:20:00             | 2 |
| 00:30:00             | 1 |
| 16:00:00             | 1 |
| 3 days 02:20:00      | 1 |

**Table 3** - Frequency of Time Intervals in the Dataset


| Index | Date Time           | Gap Duration |
|------|---------------------|--------------|
| 40378 | 2009-10-08 10:10:00 | 00:30:00 |
| 229875 | 2013-05-16 09:10:00 | 00:20:00 |
| 293229 | 2014-07-30 08:20:00 | 00:20:00 |
| 301346 | 2014-09-25 09:00:00 | 16:00:00 |
| 410940 | 2016-10-28 12:50:00 | 3 days 02:20:00 |

**Table 4** - Detected Timestamp Gaps and Their Durations


In [27]:
# Check time deltas (frequency and gaps)
deltas = df["Date Time"].diff().dropna()
print("Most common deltas:")
display(deltas.value_counts().head(5))

mode_delta = deltas.value_counts().idxmax()
print("Mode delta:", mode_delta)

# Count gaps larger than the expected frequency
gaps = deltas[deltas > mode_delta]
print("Number of gaps larger than mode delta:", len(gaps))

# Show a few gaps (if any)
if len(gaps) > 0:
    idx = gaps.index[:10]
    display(df.loc[idx, ["Date Time"]].assign(delta=gaps.loc[idx].values))

## 3.2 Seasonal Patterns

To better characterise the temporal dynamics of the dataset, seasonal patterns at different time scales are analysed. First, rolling statistics are used to assess the stationarity of the temperature series over a 7‑day window. Then, a preview of hourly resampling is performed to evaluate how aggregating 10‑minute measurements into hourly means affects the structure of the data. These analyses help to understand the dominant seasonal components and guide the choice of an appropriate modelling strategy.


### 3.2.1 Stationarity (Rolling Mean and Standard Deviation)

To assess the stationarity of the temperature series, a rolling window of 7 days was applied to compute the rolling mean and rolling standard deviation.

The results reveal clear seasonal fluctuations in the rolling mean, reflecting the annual temperature cycle. The rolling standard deviation also varies over time, indicating changes in variability across different periods of the year.

These observations suggest that the temperature series is non-stationary due to strong seasonal patterns rather than a long-term structural trend. This behaviour is typical in meteorological time series and highlights the importance of using forecasting models capable of handling seasonal and non-stationary data.

In [28]:
# Rolling mean and std for temperature (7-day window)
w = 7 * 24 * 6  # 7 days with 10-min data
t = df.set_index("Date Time")["T (degC)"]

roll_mean = t.rolling(w).mean()
roll_std = t.rolling(w).std()

plt.figure(figsize=(10, 4))
plt.plot(t, alpha=0.4, label="Temperature")
plt.plot(roll_mean, label="Rolling Mean (7 days)")
plt.plot(roll_std, label="Rolling Std (7 days)")
plt.title("Temperature - Rolling Mean and Std (7 days)")
plt.legend()
plt.show()

**Figure 65** - Temperature Rolling Mean and Standard Deviation (7‑Day Window)


### 3.2.2 Preview of Hourly Resampling

The dataset was resampled to an hourly frequency using the mean of all measurements within each hour. This aggregation reduces short-term noise and redundancy while preserving the overall temporal structure of the meteorological variables.

The resulting hourly dataset contains 70,129 observations and maintains the same set of variables as the original dataset. A verification of the timestamp sequence confirmed that no hourly timestamps are missing, indicating that the time index remains continuous after resampling.

However, some missing values appear after the aggregation step. These NaNs correspond to hours where no valid sensor measurements were available within the original 10-minute data. The number of missing values detected in each variable is summarised in Table 5.

These missing observations must be addressed before model training to ensure that the forecasting models receive complete input sequences.


| Variable | Number of NaN Values |
|--------|----------------------|
| p (mbar) | 88 |
| T (degC) | 88 |
| Tpot (K) | 88 |
| Tdew (degC) | 88 |
| rh (%) | 88 |
| VPmax (mbar) | 88 |
| VPact (mbar) | 88 |
| VPdef (mbar) | 88 |
| sh (g/kg) | 88 |
| H2OC (mmol/mol) | 88 |
| rho (g/m³) | 88 |
| wv (m/s) | 90 |
| max. wv (m/s) | 91 |
| wd (deg) | 88 |

**Table 5** - Number of Missing Values per Variable


In [29]:
feature_keys = [
    "p (mbar)", "T (degC)", "Tpot (K)", "Tdew (degC)", "rh (%)",
    "VPmax (mbar)", "VPact (mbar)", "VPdef (mbar)", "sh (g/kg)",
    "H2OC (mmol/mol)", "rho (g/m**3)", "wv (m/s)", "max. wv (m/s)", "wd (deg)"
]

df_hourly = (
    df.set_index("Date Time")[feature_keys]
      .resample("1h")   # <- lowercase h
      .mean()
      .reset_index()
)

print("Hourly shape:", df_hourly.shape)
display(df_hourly.head())

In [30]:
print("Hourly missing timestamps:", df_hourly["Date Time"].isna().sum())
print("NaNs per column (hourly):")
display(df_hourly.isna().sum())

In [31]:
end = df["Date Time"].max()
start = end - pd.Timedelta(days=7)

raw_7d = df[(df["Date Time"] >= start) & (df["Date Time"] <= end)]
hourly_7d = df_hourly[(df_hourly["Date Time"] >= start) & (df_hourly["Date Time"] <= end)]

plt.figure(figsize=(10, 4))
plt.plot(raw_7d["Date Time"], raw_7d["T (degC)"], alpha=0.4, label="Raw (10 min)")
plt.plot(hourly_7d["Date Time"], hourly_7d["T (degC)"], label="Hourly (mean)")
plt.title("Temperature: Raw vs Hourly (7 days)")
plt.xlabel("Date")
plt.ylabel("T (degC)")
plt.legend()
plt.show()

**Figure 66** - Temperature: Raw (10‑min) vs Hourly Mean (7‑Day Window)


## 3.3 Identification of Missing Hourly Observations

Rows containing missing values in the hourly dataset were identified using a row-wise NaN check. In total, 88 hourly rows contain missing values across the meteorological variables.

Inspection of these rows shows that the missing values occur simultaneously across most variables, indicating periods where the original 10-minute dataset contained no valid measurements. These rows appear in contiguous blocks, corresponding to short intervals where the weather station temporarily stopped recording data.

Since these rows contain no usable information for modelling, they must be handled before further analysis. In this study, the affected rows were removed to ensure that the resulting time series contains only complete observations.


In [32]:
na_rows = df_hourly[df_hourly.isna().any(axis=1)].copy()
print("Hourly rows with NaN:", len(na_rows))
display(na_rows.head(10))
display(na_rows.tail(10))

### 3.3.1 Removal of Missing Hourly Rows

To ensure data consistency, all hourly rows containing NaN values were removed from the dataset. This cleaning step reduced the number of observations from 70,129 to 70,038 rows.

A final verification confirmed that no missing values remain in the cleaned hourly dataset. This cleaned version of the hourly data is used in all subsequent modelling steps.

| Dataset Stage | Rows | Columns |
|---------------|------|--------|
| Hourly dataset (before cleaning) | 70,129 | 15 |
| Hourly dataset (after removing NaNs) | 70,038 | 15 |
| Remaining missing values | 0 | – |

**Table 6** - Missing Values per Variable (Hourly Dataset)


In [33]:
# Clean hourly data by dropping hours with no observations
df_hourly_clean = df_hourly.dropna().reset_index(drop=True)

print("Hourly before cleaning:", df_hourly.shape)
print("Hourly after cleaning: ", df_hourly_clean.shape)

# Sanity check
print("Remaining NaNs:", df_hourly_clean.isna().sum().sum())

# 4. Data Preparation

This section describes the steps required to prepare the dataset for modelling. The process includes selecting the relevant variables, confirming data integrity, defining the temporal frequency, setting the datetime index, splitting the data into training, validation and test sets, engineering additional time‑based features, scaling the inputs, and finally constructing sliding windows for supervised learning.


## 4.1 Dataset Scope and Columns

The forecasting target selected for this study is air temperature in degrees Celsius, T (degC).  
The input variables were defined according to the specifications of the project guidelines, which require the use of a subset of meteorological variables from the Jena Climate dataset.

All variables were extracted from the cleaned hourly-resampled dataset, ensuring that the data share a consistent temporal resolution and contain no missing values. This dataset serves as the working base for all subsequent modelling steps.

### 4.1.1 Define Target, Inputs and Frequency

The target variable is defined as the hourly air temperature T (degC).

The input variables used for forecasting include:

* Atmospheric pressure (p (mbar))
* Relative humidity (rh (%))
* Wind speed (wv (m/s))
* Maximum wind speed (max. wv (m/s))
* Wind direction (wd (deg))

These variables were extracted from the cleaned hourly dataset to construct the modelling dataframe. Using a consistent hourly frequency ensures that all predictors are temporally aligned with the target variable and suitable for time-series forecasting.


In [34]:
# Target variable
target_col = "T (degC)"

# Input features (Jena standard set)
feature_keys = ["T (degC)", "p (mbar)", "rh (%)", "wv (m/s)", "max. wv (m/s)", "wd (deg)"]

# Final working dataframe (hourly, clean)
df_work = df_hourly_clean[["Date Time"] + feature_keys].copy()

print("df_work shape:", df_work.shape)
display(df_work.head())

### 4.1.2 Sanity Checks

A validation step was performed to ensure that the working dataset contains no missing values and that all columns have the correct data types. All variables were confirmed to be numerical, and the datetime column was correctly stored as a `DatetimeIndex`. These checks guarantee that the dataset is ready for temporal indexing and model training.

In [35]:
print("NaNs per column:")
display(df_work.isna().sum())

print("\nDtypes:")
display(df_work.dtypes)

### 4.1.3 Confirm Frequency is Hourly

The time differences between consecutive observations were computed to verify the sampling frequency after the resampling step. The most common interval between observations is exactly 1 hour, confirming that the dataset is uniformly spaced in time.

A very small number of larger intervals were also detected, corresponding to previously identified data gaps in the original dataset. These rare gaps appear as longer time differences but do not affect the overall regular hourly structure of the cleaned dataset.

Maintaining a uniform temporal frequency is essential for constructing sliding windows and training sequence models such as recurrent neural networks and transformers.

| Time Difference Between Consecutive Observations | Count |
|--------------------------------|-------|
| 1 hour | 70,034 |
| 16 hours | 1 |
| 4 hours | 1 |

**Table 7** - Time Differences Between Consecutive Observations (Hourly Dataset)


In [36]:
# Check time delta between consecutive rows
deltas = df_work["Date Time"].diff().dropna()
print("Most common deltas:")
display(deltas.value_counts().head(3))

### 4.1.4 Set Date Time as Index

The `Date Time` column was set as the dataframe index, converting it into a `DatetimeIndex`. This enables efficient slicing, alignment, resampling and windowing operations, which are fundamental for time‑series forecasting workflows.

In [37]:
df_work = df_work.set_index("Date Time")
print("Index type:", type(df_work.index))
display(df_work.head())

## 4.2 Temporal Split (Train / Validation / Test)

To evaluate the forecasting models in a realistic setting, the dataset was split into training, validation and test sets using a purely temporal partition. This avoids information leakage from the future into the past and mimics real‑world deployment conditions.


### 4.2.1 Split Proportions

The dataset was divided into three subsets using the following proportions: 70% for training, 15% for validation, and 15% for testing. 

Since this is a time-series forecasting problem, the split was performed chronologically rather than randomly, ensuring that the training data precede the validation and test sets in time. This prevents information leakage from future observations.

Based on the total number of hourly samples, the resulting dataset sizes are shown in Table X. These splits provide sufficient data for model training, hyperparameter tuning, and final performance evaluation.

| Dataset Split | Number of Samples | Proportion |
|--------------|------------------|------------|
| Training set | 49,026 | 70% |
| Validation set | 10,505 | 15% |
| Test set | 10,507 | 15% |
| Total | 70,038 | 100% |

**Table 8** - Dataset Split Proportions (Train, Validation, Test)


In [38]:
# Split ratios
train_ratio = 0.70
val_ratio   = 0.15
test_ratio  = 0.15

n_total = len(df_work)
n_train = int(n_total * train_ratio)
n_val   = int(n_total * val_ratio)

print("Total samples:", n_total)
print("Train size:", n_train)
print("Val size:", n_val)
print("Test size:", n_total - n_train - n_val)

### 4.2.2 Contiguous Temporal Splits

The dataset was split into three contiguous time blocks while preserving the chronological order of the observations. The training set contains the earliest portion of the series, followed by the validation set, and finally the test set, which represents the most recent period of the dataset.

This chronological splitting strategy ensures that the model is always trained using past observations and evaluated on unseen future data, which reflects a realistic forecasting scenario. The temporal coverage of each subset is summarised in Table X, and the corresponding dataset sizes are reported in Table Y.

A visual inspection of the temperature series across the three subsets confirms that all seasonal patterns are represented and that no temporal overlap occurs between the training, validation, and test datasets.

| Dataset Split | Start Date | End Date |
|---------------|------------|----------|
| Training set | 2009-01-01 00:00 | 2014-08-05 17:00 |
| Validation set | 2014-08-05 18:00 | 2015-10-18 04:00 |
| Test set | 2015-10-18 05:00 | 2017-01-01 00:00 |

**Table 9** - Dataset Split: Temporal Ranges (Train, Validation, Test)

| Dataset Split | Rows | Columns |
|---------------|------|---------|
| Training set | 49,026 | 6 |
| Validation set | 10,505 | 6 |
| Test set | 10,507 | 6 |

**Table 10** - Dataset Split: Rows and Columns per Subset

In [39]:
df_train = df_work.iloc[:n_train]
df_val   = df_work.iloc[n_train:n_train + n_val]
df_test  = df_work.iloc[n_train + n_val:]

print("Train period:", df_train.index.min(), "->", df_train.index.max())
print("Val period:  ", df_val.index.min(),   "->", df_val.index.max())
print("Test period: ", df_test.index.min(),  "->", df_test.index.max())

print("\nShapes:")
print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)

#### 4.2.3 Visual target over splits

In [40]:
plt.figure(figsize=(10, 4))
plt.plot(df_train.index, df_train[target_col], label="Train")
plt.plot(df_val.index,   df_val[target_col],   label="Val")
plt.plot(df_test.index,  df_test[target_col],  label="Test")
plt.title("Temporal Split of Temperature")
plt.xlabel("Date")
plt.ylabel("T (degC)")
plt.legend()
plt.show()

**Figure 67** - Temporal Split of Temperature (Train, Validation, Test)


## 4.3 Feature Engineering

Additional time-based features were engineered to help the models capture periodic patterns present in the meteorological data.

From the datetime index, the hour of day** and day of year were extracted and encoded using sine and cosine transformations. This cyclical encoding preserves the circular nature of time variables, ensuring that values such as 23:00 and 00:00 remain close in the feature space. These features allow the model to capture daily and annual seasonality.

Wind direction is also a circular variable ranging from 0° to 360°. Therefore, it was transformed into sine and cosine components to correctly represent its angular structure. The original wind direction variable was then removed to avoid redundancy.

After applying these transformations, the resulting feature set contains 11 variables, including the original meteorological variables and the newly engineered cyclical features. These additional predictors provide explicit representations of periodic behaviour that can improve the performance of sequence forecasting models.

| Feature | Description |
|-------|-------------|
| T (degC) | Target temperature |
| p (mbar) | Atmospheric pressure |
| rh (%) | Relative humidity |
| wv (m/s) | Wind speed |
| max. wv (m/s) | Maximum wind speed |
| hour_sin | Sine encoding of hour of day |
| hour_cos | Cosine encoding of hour of day |
| doy_sin | Sine encoding of day of year |
| doy_cos | Cosine encoding of day of year |
| wd_sin | Sine encoding of wind direction |
| wd_cos | Cosine encoding of wind direction |

**Table 11** - Features Used and Their Descriptions


In [41]:
def add_time_features(df_in):
    df = df_in.copy()

    # time parts
    hour = df.index.hour
    doy = df.index.dayofyear

    # cyclical encoding
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    df["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

    # wind direction is circular
    wd = df["wd (deg)"].astype(float)
    df["wd_sin"] = np.sin(2 * np.pi * wd / 360.0)
    df["wd_cos"] = np.cos(2 * np.pi * wd / 360.0)

    # drop original angular variable
    df.drop(columns=["wd (deg)"], inplace=True)

    return df

df_train_fe = add_time_features(df_train)
df_val_fe   = add_time_features(df_val)
df_test_fe  = add_time_features(df_test)

print(df_train_fe.shape, df_val_fe.shape, df_test_fe.shape)

## 4.4 Feature Scaling

All input features, including the engineered time-based variables, were standardised using the `StandardScaler` method. This transformation rescales each feature so that it has zero mean and unit variance, which helps stabilise gradient-based optimisation during neural network training.

To prevent data leakage, the scaler was fitted exclusively on the training dataset. The same transformation parameters were then applied to the validation and test sets, ensuring that no information from future observations influenced the training process.

Feature scaling is particularly important for deep learning models such as recurrent neural networks and transformers, as it ensures that variables with different numerical ranges contribute proportionally during model optimisation.

A final verification confirmed that no NaN values were introduced during the scaling process.

In [42]:
from sklearn.preprocessing import StandardScaler

# Choose features to scale (all columns)
feat_cols = df_train_fe.columns.tolist()

scaler = StandardScaler()
scaler.fit(df_train_fe[feat_cols])

train_scaled = scaler.transform(df_train_fe[feat_cols])
val_scaled   = scaler.transform(df_val_fe[feat_cols])
test_scaled  = scaler.transform(df_test_fe[feat_cols])

# Back to DataFrame (keep index)
df_train_s = pd.DataFrame(train_scaled, index=df_train_fe.index, columns=feat_cols)
df_val_s   = pd.DataFrame(val_scaled,   index=df_val_fe.index,   columns=feat_cols)
df_test_s  = pd.DataFrame(test_scaled,  index=df_test_fe.index,  columns=feat_cols)

print("NaNs after scaling:", df_train_s.isna().sum().sum(), df_val_s.isna().sum().sum(), df_test_s.isna().sum().sum())

## 4.5 Windowing (L = 120, H = 24)

To cast the forecasting problem into a supervised learning setting, sliding windows were constructed over the scaled time series data. Each input sample consists of the previous L = 120 hours (5 days) of observations, while the corresponding target consists of the next H = 24 hours of air temperature.

This formulation defines a multivariate-input, univariate-output, multi-step forecasting problem, where the model receives a sequence of past meteorological observations and directly predicts the full 24-hour temperature profile ahead.

Since the windowing procedure was applied after feature scaling, both the input sequences and the target values were represented in the scaled space during training. This is beneficial for neural network optimisation, as it stabilises the magnitude of the learning signal.

The resulting datasets are summarised in Table X.

| Dataset | X Shape | y Shape |
|---------|---------|---------|
| Training | (48,882, 120, 11) | (48,882, 24) |
| Validation | (10,361, 120, 11) | (10,361, 24) |
| Test | (10,363, 120, 11) | (10,363, 24) |

**Table 12** - Dataset Shapes After Windowing 


In [43]:
from numpy.lib.stride_tricks import sliding_window_view

L = 120  # input window (hours)
H = 24   # forecast horizon (hours)

def make_windows(df_scaled, target_name, L, H):
    # Cast to float32 once here — avoids repeated float64→float32
    # conversion inside TensorFlow on every batch during training.
    data = df_scaled.values.astype(np.float32)
    target_idx = df_scaled.columns.get_loc(target_name)
    n_samples = len(data) - L - H

    # Build X windows fully vectorized using stride tricks — no Python loop.
    # sliding_window_view(data, L, axis=0) → (n-L+1, n_features, L)
    # transpose(0,2,1)                     → (n-L+1, L, n_features)
    # [:n_samples]                          → (n_samples, L, n_features)
    X = sliding_window_view(data, window_shape=L, axis=0).transpose(0, 2, 1)[:n_samples].copy()

    # Build y windows: target column only, H steps ahead of each X window.
    # sliding_window_view on target[L:] → (n-L-H+1, H), take [:n_samples]
    y = sliding_window_view(data[L:, target_idx], window_shape=H)[:n_samples].copy()

    return X, y

X_train, y_train = make_windows(df_train_s, target_col, L, H)
X_val,   y_val   = make_windows(df_val_s,   target_col, L, H)
X_test,  y_test  = make_windows(df_test_s,  target_col, L, H)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)
print("dtype:", X_train.dtype)

### 4.5.1 Baseline: Persistence

As a simple baseline, a persistence model was implemented. For each window, the model predicts that the temperature for all 24 forecast hours will be equal to the last observed temperature in the input window. Although naive, this baseline provides a meaningful lower bound on performance and helps to contextualise the results of more complex models.

In [44]:
# persistence: predict next 24 hours as last observed temperature in the input window
target_idx = df_test_s.columns.get_loc(target_col)

last_T = X_test[:, -1, target_idx]  # last temp in each window
y_pred_persist = np.repeat(last_T[:, None], y_test.shape[1], axis=1)

print(y_pred_persist.shape)

### 4.5.2 Evaluation Metrics

The performance of the multi‑horizon forecasts was evaluated using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE), computed over all horizons and all samples. These metrics quantify the average magnitude of the prediction errors and penalise larger deviations more strongly in the case of RMSE. The persistence baseline achieves an MAE of approximately 0.36 (scaled units) and an RMSE of about 0.49, which serves as a reference for subsequent models.


In [45]:
def eval_multihorizon(y_true, y_pred):
    mae = mean_absolute_error(y_true.reshape(-1), y_pred.reshape(-1))
    rmse = np.sqrt(mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1)))
    return mae, rmse

mae_p, rmse_p = eval_multihorizon(y_test, y_pred_persist)
print("Persistence - MAE:", mae_p, "RMSE:", rmse_p)

### 4.5.3 MAE by Horizon

To analyse how forecast accuracy degrades with time, the MAE was computed separately for each forecast horizon from 1 to 24 hours ahead. The error profile shows that the MAE increases for intermediate horizons and then slightly decreases for the farthest horizons, reflecting the interaction between daily cycles and the persistence assumption. This horizon‑wise analysis provides a more detailed view of model behaviour than a single aggregated metric.

In [46]:
mae_by_h = np.mean(np.abs(y_test - y_pred_persist), axis=0)

plt.figure()
plt.plot(range(1, len(mae_by_h) + 1), mae_by_h)
plt.title("Persistence baseline - MAE by horizon")
plt.xlabel("Horizon (hours ahead)")
plt.ylabel("MAE (scaled)")
plt.show()

**Figure 68** - Persistence Baseline: MAE by Forecast Horizon


# 5. GRU Model

This section describes the construction, configuration, and optimisation of a GRU-based neural network for multi-horizon temperature forecasting. The workflow includes GPU setup, reproducibility settings, model architecture definition, training procedures, and hyperparameter optimisation.

Hyperparameter optimisation was performed using Optuna, an automatic hyperparameter search framework designed for efficient optimisation of machine learning models. Optuna employs a sampling strategy that explores the hyperparameter space iteratively, evaluating model performance and updating its search strategy based on previous trials.

During the optimisation process, multiple training configurations are evaluated by varying parameters such as the number of GRU units, number of layers, dropout rate, learning rate, batch size, and regularisation strength. Each configuration corresponds to a trial, where the model is trained and evaluated on the validation set.

Optuna uses techniques such as Tree-structured Parzen Estimators (TPE) to guide the search towards promising regions of the hyperparameter space, allowing it to identify high-performing configurations more efficiently than traditional grid search methods.

The objective function minimises the validation loss, and the best-performing configuration is selected as the final model setup for subsequent training and evaluation.

| Hyperparameter | Description |
|----------------|-------------|
| units | Number of GRU units |
| n_layers | Number of GRU layers |
| dropout | Dropout rate for regularisation |
| recurrent_dropout | Dropout applied to recurrent connections |
| learning_rate | Learning rate of the optimiser |
| batch_size | Number of samples per training batch |
| l2 | L2 weight regularisation |

**Table 13** - GRU Hyperparameters and Their Descriptions


In [47]:
print("TF version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)

if gpus:
    # Avoid TF grabbing all GPU memory at once
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU is available and memory growth is enabled.")
else:
    print("No GPU found. Training will run on CPU.")

## 5.1 Setup: Reproducibility

To ensure consistent and repeatable results across runs, random seeds were fixed for both TensorFlow and NumPy. The input window length (L), forecast horizon (H) and number of features were also extracted from the training data to parameterise the model architecture.

In [48]:
tf.random.set_seed(42)
np.random.seed(42)

H = y_train.shape[1]
L = X_train.shape[1]
n_features = X_train.shape[2]

## 5.2 Model Architecture and Hyperparameters

A flexible GRU-based architecture was implemented to model the temporal dependencies in the meteorological time series. The network supports between one and three stacked GRU layers, allowing the model capacity to be adjusted depending on the hyperparameter configuration.

Each GRU layer processes sequential input data and learns temporal representations from the past observations. Dropout regularisation is applied within the recurrent layers to reduce overfitting, while optional L2 weight regularisation can be used to constrain the model parameters.

An optional Gaussian noise layer can be applied to the inputs during training, which acts as a form of regularisation by improving model robustness to small perturbations in the input data.

The architecture also supports an optional dense layer before the output layer, enabling additional non-linear feature transformations. Different activation functions can be used in this layer depending on the configuration explored during hyperparameter optimisation.

The final output layer produces a vector of length H = 24, corresponding to the 24-hour forecasting horizon. This implements a direct multi-step forecasting strategy, where the model predicts the entire temperature trajectory for the next day in a single forward pass.

Model training is performed using gradient-based optimisation with either the Adam or AdamW optimiser. Gradient clipping is applied to stabilise training, and several loss functions can be selected (MSE, MAE, or Huber loss) depending on the optimisation configuration.

| Hyperparameter | Description |
|----------------|-------------|
| units1, units2, units3 | Number of GRU units per layer |
| n_layers | Number of stacked GRU layers |
| dropout | Dropout rate applied to recurrent layers |
| l2 | L2 weight regularisation strength |
| dense_units | Size of optional dense layer |
| dense_activation | Activation function in dense layer |
| learning_rate | Learning rate of the optimiser |
| optimizer_name | Optimiser type (Adam or AdamW) |
| weight_decay | Weight decay parameter for AdamW |
| clipnorm | Gradient clipping threshold |
| loss_name | Training loss function (MSE, MAE, Huber) |
| gaussian_noise_std | Standard deviation of input Gaussian noise |

**Table 14** - Hyperparameters and Their Descriptions

In [49]:
def build_gru_model(
    L, n_features, H,
    units1=64,
    units2=32,
    n_layers=1,
    dropout=0.2,
    l2=0.0,
    dense_units=0,
    dense_activation="relu",
    learning_rate=1e-3,
    clipnorm=1.0,
    optimizer_name="adam",
    # --- new (optional) params, defaults keep old behavior ---
    units3=32,
    weight_decay=1e-5,          # used only if adamw
    loss_name="mse",            # mse/mae/huber1/huber2
    gaussian_noise_std=0.0,     # 0 disables
):
    reg = keras.regularizers.l2(l2) if l2 and l2 > 0 else None

    inputs = keras.Input(shape=(L, n_features))
    x = inputs

    # Optional input noise
    if gaussian_noise_std and gaussian_noise_std > 0:
        x = layers.GaussianNoise(gaussian_noise_std)(x)

    # GRU stack (supports 1/2/3 layers).
    # recurrent_dropout=0.0 is required to use the fast cuDNN GRU kernel on GPU.
    x = layers.GRU(
        units1,
        return_sequences=(n_layers >= 2),
        dropout=dropout,
        recurrent_dropout=0.0,
        kernel_regularizer=reg
    )(x)

    if n_layers >= 2:
        x = layers.GRU(
            units2,
            return_sequences=(n_layers == 3),
            dropout=dropout,
            recurrent_dropout=0.0,
            kernel_regularizer=reg
        )(x)

    if n_layers == 3:
        x = layers.GRU(
            units3,
            return_sequences=False,
            dropout=dropout,
            recurrent_dropout=0.0,
            kernel_regularizer=reg
        )(x)

    # Optional dense head with extra activations
    if dense_units and dense_units > 0:
        if dense_activation == "leaky_relu":
            x = layers.Dense(dense_units, kernel_regularizer=reg)(x)
            x = layers.LeakyReLU(negative_slope=0.1)(x)
        else:
            x = layers.Dense(dense_units, activation=dense_activation, kernel_regularizer=reg)(x)
        x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(H)(x)  # linear output
    model = keras.Model(inputs, outputs)

    # Optimizer
    if optimizer_name.lower() == "adamw":
        opt = keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            clipnorm=clipnorm
        )
    else:
        opt = keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=clipnorm)

    # Loss
    if loss_name == "mae":
        loss = "mae"
    elif loss_name == "huber1":
        loss = keras.losses.Huber(delta=1.0)
    elif loss_name == "huber2":
        loss = keras.losses.Huber(delta=2.0)
    else:
        loss = "mse"

    model.compile(optimizer=opt, loss=loss, metrics=["mae"])
    return model

In [50]:
def build_gru_model(
    L, n_features, H,
    units1=64,
    units2=32,
    n_layers=1,
    dropout=0.2,
    l2=0.0,
    dense_units=0,
    dense_activation="relu",
    learning_rate=1e-3,
    clipnorm=1.0,
    optimizer_name="adam",
    # --- new (optional) params, defaults keep old behavior ---
    units3=32,
    weight_decay=1e-5,          # used only if adamw
    loss_name="mse",            # mse/mae/huber1/huber2
    gaussian_noise_std=0.0,     # 0 disables
):
    reg = keras.regularizers.l2(l2) if l2 and l2 > 0 else None

    inputs = keras.Input(shape=(L, n_features))
    x = inputs

    # Optional input noise
    if gaussian_noise_std and gaussian_noise_std > 0:
        x = layers.GaussianNoise(gaussian_noise_std)(x)

    # GRU stack (supports 1/2/3 layers).
    # recurrent_dropout=0.0 is required to use the fast cuDNN GRU kernel on GPU.
    x = layers.GRU(
        units1,
        return_sequences=(n_layers >= 2),
        dropout=dropout,
        recurrent_dropout=0.0,
        kernel_regularizer=reg
    )(x)

    if n_layers >= 2:
        x = layers.GRU(
            units2,
            return_sequences=(n_layers == 3),
            dropout=dropout,
            recurrent_dropout=0.0,
            kernel_regularizer=reg
        )(x)

    if n_layers == 3:
        x = layers.GRU(
            units3,
            return_sequences=False,
            dropout=dropout,
            recurrent_dropout=0.0,
            kernel_regularizer=reg
        )(x)

    # Optional dense head with extra activations
    if dense_units and dense_units > 0:
        if dense_activation == "leaky_relu":
            x = layers.Dense(dense_units, kernel_regularizer=reg)(x)
            x = layers.LeakyReLU(negative_slope=0.1)(x)
        else:
            x = layers.Dense(dense_units, activation=dense_activation, kernel_regularizer=reg)(x)
        x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(H)(x)  # linear output
    model = keras.Model(inputs, outputs)

    # Optimizer
    if optimizer_name.lower() == "adamw":
        opt = keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            clipnorm=clipnorm
        )
    else:
        opt = keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=clipnorm)

    # Loss
    if loss_name == "mae":
        loss = "mae"
    elif loss_name == "huber1":
        loss = keras.losses.Huber(delta=1.0)
    elif loss_name == "huber2":
        loss = keras.losses.Huber(delta=2.0)
    else:
        loss = "mse"

    model.compile(optimizer=opt, loss=loss, metrics=["mae"])
    return model

## 5.3 Training Wrapper and Callbacks

To facilitate experimentation with multiple hyperparameter configurations, a dedicated training wrapper function was implemented. This function builds the GRU model according to a given configuration, trains the model, and returns the resulting performance metrics.

The wrapper enables a consistent training procedure across all experiments by automatically constructing the model architecture, applying the selected hyperparameters, and executing the training process.

Two callback mechanisms are used during training to improve optimisation stability and prevent overfitting:

* EarlyStopping, which monitors the validation loss and stops training if no improvement is observed for several epochs. The best model weights are automatically restored.
* ReduceLROnPlateau, which reduces the learning rate when the validation loss stagnates, allowing the optimiser to refine the solution.

After training, the function extracts the best validation metrics obtained during the training process, including the minimum validation loss and the corresponding validation MAE. These metrics are used to compare different configurations during the hyperparameter optimisation stage.

This modular training setup allows efficient evaluation of multiple hyperparameter configurations during the Optuna optimisation process.

In [51]:
def train_one_config(cfg, X_train, y_train, X_val, y_val, max_epochs=60, fit_verbose=0):
    model = build_gru_model(
        L=L, n_features=n_features, H=H,
        units1=cfg["units1"],
        units2=cfg["units2"],
        n_layers=cfg["n_layers"],
        dropout=cfg["dropout"],
        l2=cfg["l2"],
        dense_units=cfg["dense_units"],
        dense_activation=cfg["dense_activation"],
        learning_rate=cfg["lr"],
        clipnorm=cfg["clipnorm"],
        optimizer_name=cfg["optimizer"],
        # new params
        units3=cfg.get("units3", 32),
        weight_decay=cfg.get("weight_decay", 0.0),
        loss_name=cfg.get("loss_name", "mse"),
        gaussian_noise_std=cfg.get("gaussian_noise_std", 0.0),
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=max_epochs,
        batch_size=cfg["batch_size"],
        verbose=fit_verbose,
        callbacks=callbacks
    )

    best_val_loss = float(np.min(history.history["val_loss"]))
    best_val_mae  = float(np.min(history.history["val_mae"]))
    return model, history, best_val_loss, best_val_mae

## 5.4 Hyperparameter Optimisation with Optuna

Hyperparameter optimisation was performed using the Optuna framework, which implements efficient Bayesian optimisation for machine learning models. Optuna uses a Tree-structured Parzen Estimator (TPE) sampler to explore the hyperparameter space and identify promising configurations based on previous trials.

During the optimisation process, each trial samples a set of architectural and training hyperparameters, including the number of GRU layers, number of units per layer, dropout rate, learning rate, batch size, optimiser type, regularisation strength, and loss function.

A custom pruning callback was implemented to integrate Optuna with the Keras training loop. After each epoch, the validation loss is reported to Optuna, allowing the Hyperband pruner to terminate trials that are unlikely to outperform the current best configuration. This significantly reduces computational cost by avoiding unnecessary training of poorly performing models.

For each trial, the GRU model is trained using the sampled hyperparameters and evaluated on the validation dataset. The objective function minimises the validation loss, while the corresponding validation MAE is also recorded for analysis.

The study continuously tracks the best-performing configuration and stores the associated model, training history, and hyperparameter settings. This enables the final model selection to be based on the configuration achieving the lowest validation loss during the optimisation process.

| Hyperparameter | Values Explored |
|----------------|----------------|
| n_layers | 1, 2, 3 |
| units1 | 64, 96, 128, 192 |
| units2 | 32, 64, 96, 128 |
| units3 | 32, 64, 96 |
| dropout | 0.0, 0.1, 0.2, 0.3, 0.4 |
| l2 | 0.0, 1e-6, 1e-5, 1e-4 |
| dense_units | 0, 64, 128, 256 |
| dense_activation | relu, gelu, elu, leaky_relu |
| learning_rate | 2e-3, 1e-3, 5e-4, 3e-4, 2e-4, 1e-4 |
| batch_size | 128, 256, 512 |
| clipnorm | 0.5, 1.0, 2.0, 5.0 |
| optimizer | adam, adamw |
| weight_decay | 0.0, 1e-6, 1e-5, 1e-4 |
| loss_name | mse, mae, huber1, huber2 |
| gaussian_noise_std | 0.0, 0.01, 0.05 |

**Table 15** - Hyperparameters and Search Space Explored with Optuna

In [52]:
# Custom pruning callback — avoids dependency on optuna.integration.*
# After each epoch it reports val_loss to Optuna; if Optuna's Hyperband pruner
# decides this trial is unlikely to beat the current best, it raises TrialPruned
# and Keras stops training immediately, saving compute.
class OptunaPruningCallback(keras.callbacks.Callback):
    def __init__(self, trial):
        super().__init__()
        self.trial = trial

    def on_epoch_end(self, epoch, logs=None):
        val_loss = logs.get("val_loss")
        if val_loss is None:
            return
        self.trial.report(val_loss, epoch)
        if self.trial.should_prune():
            raise optuna.exceptions.TrialPruned()


# Mutable dict to keep the best Keras model across trials.
# Optuna tracks hyperparameters and val_loss, but not model objects.
best_holder = {"val_loss": np.inf, "model": None, "cfg": None, "history": None}


def objective(trial):
    # ── Sample hyperparameters via TPE (Bayesian) ────────────────────────────
    n_layers    = trial.suggest_int("n_layers", 1, 3)
    units1      = trial.suggest_categorical("units1",      [64, 96, 128, 192])
    units2      = trial.suggest_categorical("units2",      [32, 64, 96, 128])
    units3      = trial.suggest_categorical("units3",      [32, 64, 96])
    dropout     = trial.suggest_categorical("dropout",     [0.0, 0.1, 0.2, 0.3, 0.4])
    l2          = trial.suggest_categorical("l2",          [0.0, 1e-6, 1e-5, 1e-4])
    dense_units = trial.suggest_categorical("dense_units", [0, 64, 128, 256])
    dense_act   = trial.suggest_categorical("dense_activation", ["relu", "gelu", "elu", "leaky_relu"])
    lr          = trial.suggest_categorical("lr",          [2e-3, 1e-3, 5e-4, 3e-4, 2e-4, 1e-4])
    batch_size  = trial.suggest_categorical("batch_size",  [128, 256, 512])
    clipnorm    = trial.suggest_categorical("clipnorm",    [0.5, 1.0, 2.0, 5.0])
    opt_name    = trial.suggest_categorical("optimizer",   ["adam", "adamw"])
    weight_decay= trial.suggest_categorical("weight_decay",[0.0, 1e-6, 1e-5, 1e-4])
    loss_name   = trial.suggest_categorical("loss_name",   ["mse", "mae", "huber1", "huber2"])
    noise_std   = trial.suggest_categorical("gaussian_noise_std", [0.0, 0.01, 0.05])

    # ── Apply same constraints as the random search ──────────────────────────
    if units2 > units1:   units2 = units1
    if units3 > units2:   units3 = units2
    if opt_name == "adam": weight_decay = 0.0
    if n_layers == 1 and dense_units > 0 and random.random() < 0.7:
        dense_units = 0

    cfg = dict(
        n_layers=n_layers, units1=units1, units2=units2, units3=units3,
        dropout=dropout, l2=l2, dense_units=dense_units, dense_activation=dense_act,
        lr=lr, batch_size=batch_size, clipnorm=clipnorm, optimizer=opt_name,
        weight_decay=weight_decay, loss_name=loss_name, gaussian_noise_std=noise_std,
    )

    model = build_gru_model(
        L=L, n_features=n_features, H=H,
        units1=units1, units2=units2, units3=units3,
        n_layers=n_layers, dropout=dropout, l2=l2,
        dense_units=dense_units, dense_activation=dense_act,
        learning_rate=lr, clipnorm=clipnorm,
        optimizer_name=opt_name, weight_decay=weight_decay,
        loss_name=loss_name, gaussian_noise_std=noise_std,
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
        OptunaPruningCallback(trial),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=60,
        batch_size=batch_size,
        verbose=0,
        callbacks=callbacks,
    )

    val_loss = float(np.min(history.history["val_loss"]))
    val_mae  = float(np.min(history.history["val_mae"]))

    trial.set_user_attr("val_mae", val_mae)

    if val_loss < best_holder["val_loss"]:
        best_holder.update({"val_loss": val_loss, "model": model, "history": history, "cfg": cfg})

    return val_loss


## 5.5 Hyperparameter Optimisation Run

The hyperparameter optimisation was executed using Optuna with a Tree-structured Parzen Estimator (TPE) sampler and a Hyperband pruning strategy*. The TPE sampler performs Bayesian optimisation by modelling the distribution of promising hyperparameter configurations based on previously evaluated trials, allowing the search process to focus on regions of the parameter space likely to produce better results.

To reduce computational cost, the Hyperband pruner was used to terminate unpromising trials early. Each trial was required to train for a minimum number of epochs before pruning decisions were made, ensuring that early training noise did not prematurely eliminate potentially good configurations.

A custom callback was implemented to provide live visual feedback during the optimisation process, displaying the validation loss progression across completed trials as well as the duration of each trial. This allowed real-time monitoring of optimisation progress and computational efficiency.

In total, 150 trials were executed. Among these, 92 trials completed successfully, while 58 trials were pruned early by the Hyperband strategy. The best configuration achieved a validation loss of 0.0337 (MSE on scaled data).

| Metric | Value |
|------|------|
| Total trials | 150 |
| Completed trials | 92 |
| Pruned trials | 58 |
| Best validation loss | 0.0337 |
| Optimisation method | TPE (Bayesian) |
| Pruning strategy | Hyperband |

**Table 16** - Summary of the Hyperparameter Optimisation Run


The optimisation results show that the pruning strategy effectively reduced the computational cost by terminating nearly 40% of trials early, while still identifying high-performing configurations within the search space.

In [53]:
from IPython.display import clear_output

def progress_callback(study, trial):
    """Live chart redrawn after every trial (complete or pruned)."""
    clear_output(wait=True)
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned    = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    n_done    = len(study.trials)

    state_str = ("PRUNED" if trial.state == optuna.trial.TrialState.PRUNED
                 else f"val_loss={trial.value:.4f}")
    print(f"[{n_done}/150] Trial #{trial.number} → {state_str}")
    if completed:
        print(f"  Complete: {len(completed)} | Pruned: {len(pruned)} | Best so far: {study.best_value:.4f}")

    if len(completed) < 2:
        return

    losses       = [t.value for t in completed]
    running_best = pd.Series(losses).cummin()
    durations    = [t.duration.total_seconds() for t in completed if t.duration]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.scatter(range(1, len(losses) + 1), losses, alpha=0.5, s=30, label="Trial val_loss")
    ax.plot(range(1, len(losses) + 1), running_best, color="red", linewidth=2, label="Running best")
    ax.set_title(f"Optuna Progress [{len(completed)} complete, {len(pruned)} pruned]"
                 f" | Best: {study.best_value:.4f}")
    ax.set_xlabel("Completed trial index")
    ax.set_ylabel("val_loss (MSE, scaled)")
    ax.legend()

    ax = axes[1]
    ax.bar(range(1, len(durations) + 1), durations, color="steelblue", alpha=0.7)
    if durations:
        ax.axhline(np.mean(durations), color="red", linestyle="--",
                   label=f"Mean: {np.mean(durations):.1f}s")
    ax.set_title("Trial Duration")
    ax.set_xlabel("Completed trial index")
    ax.set_ylabel("Duration (s)")
    ax.legend()

    plt.tight_layout()
    plt.show()


# TPE sampler: Bayesian optimisation via Tree-structured Parzen Estimators.
# After an initial random phase (~10 trials) it models the loss landscape
# and focuses sampling on promising regions — unlike pure random search.
#
# HyperbandPruner: kills unpromising trials early based on epoch-level val_loss.
# min_resource=5  → every trial runs at least 5 epochs before pruning kicks in.
# max_resource=60 → matches our max_epochs.
# reduction_factor=3 → each bracket keeps the top 1/3 of trials.
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.HyperbandPruner(min_resource=5, max_resource=60, reduction_factor=3),
)

study.optimize(objective, n_trials=150, callbacks=[progress_callback])

print("\nOptimization complete!")
print(f"Best trial #{study.best_trial.number} — val_loss: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


**Figure 69** - Optuna Optimisation Progress and Trial Durations

## 5.6 Best Model Configuration

After completing the hyperparameter optimisation process, the best-performing configuration was obtained in trial 113, achieving a validation loss of 0.0337 (MSE on scaled data).

The optimal model uses a two-layer GRU architecture with moderate hidden sizes and no dropout regularisation. Instead, a small amount of L2 weight regularisation and AdamW weight decay are used to control overfitting. An additional dense layer with 256 units is applied before the output layer to increase the model's capacity for non-linear feature transformations.

The model was trained using the AdamW optimiser with a low learning rate and Huber loss, which provides robustness to occasional large prediction errors.

The full set of hyperparameters for the best configuration is shown in Table 17.

| Hyperparameter | Best Value |
|----------------|-----------|
| n_layers | 2 |
| units1 | 96 |
| units2 | 64 |
| units3 | 96 |
| dropout | 0.0 |
| l2 | 1e-6 |
| dense_units | 256 |
| dense_activation | relu |
| learning_rate | 0.0002 |
| batch_size | 128 |
| clipnorm | 2.0 |
| optimizer | adamw |
| weight_decay | 1e-6 |
| loss_function | huber (δ = 1) |
| gaussian_noise_std | 0.0 |

**Table 17** - Best Hyperparameter Configuration

The results suggest that a moderately sized GRU architecture is sufficient to capture the temporal dynamics of the meteorological variables. The optimisation process favoured deeper representations combined with a dense projection layer rather than heavy recurrent regularisation.

The selection of Huber loss indicates that robustness to occasional extreme temperature variations may improve forecasting stability compared to purely quadratic loss functions.

### 5.6.1 Analysis of Hyperparameter Trials

Beyond identifying the best-performing configuration, it is important to analyse how different hyperparameter choices influence model performance. The Optuna optimisation process evaluated a wide range of architectural and training configurations, allowing several patterns to emerge across the completed trials.

Out of the 150 optimisation trials executed, 92 trials completed successfully while 58 were pruned early. Approximately 39% of trials were pruned before completing training, indicating that the Hyperband pruning strategy effectively reduced computational cost by terminating configurations that were unlikely to outperform the current best model.

* Architecture Depth: Trials with two GRU layers generally performed better than single-layer models, suggesting that additional temporal abstraction helps capture complex patterns in the meteorological time series. However, increasing the depth to three layers did not consistently improve performance, likely due to increased model complexity and training instability.
* Number of Units: Configurations with moderate hidden sizes (e.g., 96–64 units) tended to outperform both very small and very large architectures. Smaller models may lack sufficient capacity to model temporal dynamics, while excessively large models risk overfitting or slower convergence.
* Regularisation: Interestingly, the best-performing configuration used no dropout, but relied on mild L2 regularisation together with weight decay through the AdamW optimiser. This suggests that with a sufficiently large dataset, heavy dropout regularisation may not be necessary to control overfitting.
* Loss Functions: Different loss functions were explored during optimisation. Trials using Huber loss frequently achieved competitive results, likely because this loss function is less sensitive to occasional large prediction errors caused by extreme temperature fluctuations.
* Learning Rate and Optimiser: Lower learning rates (around 2 × 10⁻⁴) consistently produced more stable training behaviour. The AdamW optimiser also appeared in several strong-performing trials, indicating that decoupled weight decay can provide more stable optimisation than standard Adam in some configurations.
* Dense Projection Layer: Many of the best-performing trials included an additional dense layer before the output. This layer enables further non-linear transformations before producing the 24-hour forecast vector, improving the model's ability to capture complex relationships between the input variables and the predicted temperature trajectory.

Taken together, the optimisation experiments indicate that moderately sized GRU architectures combined with stable optimisation settings provide reliable forecasting performance for this task.

| Rank | Trial | Best Val Loss | Best Val MAE | Train Time (s) | Layers | Units1 | Units2 | Units3 | Dropout | L2 | Dense Units | Dense Act | LR | Batch | Clipnorm | Optimizer | Weight Decay | Loss | Noise |
|-----|------|---------------|--------------|---------------|-------|-------|-------|-------|--------|------|-----------|-----------|------|------|----------|-----------|--------------|------|------|
| 1 | trial_113 | 0.033656 | 0.192381 | 156.4 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | relu | 2e-4 | 128 | 2.0 | adamw | 1e-6 | huber1 | 0.0 |
| 2 | trial_132 | 0.033723 | 0.191858 | 161.8 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | relu | 2e-4 | 128 | 5.0 | adamw | 1e-6 | huber1 | 0.0 |
| 3 | trial_121 | 0.033961 | 0.192169 | 182.0 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | relu | 2e-4 | 128 | 5.0 | adamw | 1e-6 | huber1 | 0.0 |
| 4 | trial_115 | 0.034121 | 0.193393 | 154.7 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | relu | 2e-4 | 128 | 5.0 | adamw | 1e-6 | huber1 | 0.0 |
| 5 | trial_123 | 0.034135 | 0.193520 | 145.9 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | relu | 2e-4 | 128 | 5.0 | adamw | 1e-6 | huber1 | 0.0 |
| 6 | trial_128 | 0.034169 | 0.192650 | 169.8 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | relu | 2e-4 | 128 | 5.0 | adamw | 1e-6 | huber1 | 0.0 |
| 7 | trial_147 | 0.034188 | 0.193148 | 146.6 | 2 | 128 | 64 | 96 | 0.0 | 1e-6 | 256 | leaky_relu | 2e-4 | 128 | 5.0 | adamw | 1e-6 | huber1 | 0.0 |
| 8 | trial_111 | 0.034245 | 0.193756 | 146.4 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | relu | 2e-4 | 128 | 2.0 | adamw | 1e-6 | huber1 | 0.0 |
| 9 | trial_142 | 0.034247 | 0.192640 | 179.9 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | leaky_relu | 2e-4 | 128 | 5.0 | adamw | 1e-6 | huber1 | 0.0 |
|10 | trial_141 | 0.034269 | 0.193092 | 159.8 | 2 | 96 | 64 | 96 | 0.0 | 1e-6 | 256 | leaky_relu | 2e-4 | 128 | 5.0 | adamw | 1e-6 | huber1 | 0.0 |

**Table 18** - Best Hyperparameter Values from Optuna Optimisation

In [ ]:
# Build results_df from completed Optuna trials in the same column format
# as the random search version so all downstream evaluation cells work unchanged.
rows = []
for t in study.trials:
    if t.state != optuna.trial.TrialState.COMPLETE:
        continue
    row = {
        "name":           f"trial_{t.number:02d}",
        "best_val_loss":  t.value,
        "best_val_mae":   t.user_attrs.get("val_mae", np.nan),
        "train_time_sec": round(t.duration.total_seconds(), 1) if t.duration else np.nan,
        **t.params,
    }
    rows.append(row)

n_pruned = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
results_df = pd.DataFrame(rows).sort_values("best_val_loss").reset_index(drop=True)

print(f"Completed trials: {len(rows)} | Pruned: {n_pruned}")
display(results_df.head(10))

# Compatibility aliases — downstream cells reference these names.
# Inject "name" into cfg so title labels in evaluation cells work unchanged.
best_cfg_named = {**best_holder["cfg"], "name": f"trial_{study.best_trial.number:02d}"}
best = {
    "val_loss": best_holder["val_loss"],
    "model":    best_holder["model"],
    "history":  best_holder["history"],
    "cfg":      best_cfg_named,
}
best_gru = best_holder["model"]

## 5.7 Training Curves (Best GRU)

The training dynamics of the best-performing GRU model (trial #113) were analysed using the evolution of the training and validation metrics across epochs.

Both the loss (MSE) and MAE curves show a rapid decrease during the first few epochs, indicating that the model quickly learns the main temporal patterns present in the meteorological time series. After this initial phase, the improvement becomes more gradual, eventually stabilising after approximately 15–18 epochs.

The validation curves closely follow the training curves throughout the optimisation process, with only a small gap between them. This behaviour suggests that the model generalises well to unseen data and does not exhibit strong signs of overfitting.

Additionally, the validation loss reaches a plateau toward the final epochs, which indicates that the model has reached a stable solution and that further training would likely provide only marginal improvements. This behaviour is consistent with the early stopping strategy used during training.

The training curves confirm that the selected hyperparameter configuration leads to stable optimisation and reliable convergence behaviour for the GRU forecasting model.

The smooth behaviour of the curves also indicates that the chosen learning rate and gradient clipping settings provided stable optimisation without oscillations during training.

In [54]:
hist = best["history"].history

plt.figure()
plt.plot(hist["loss"], label="train_loss")
plt.plot(hist["val_loss"], label="val_loss")
plt.title(f"Best GRU - Loss ({best['cfg']['name']})")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

plt.figure()
plt.plot(hist["mae"], label="train_mae")
plt.plot(hist["val_mae"], label="val_mae")
plt.title(f"Best GRU - MAE ({best['cfg']['name']})")
plt.xlabel("Epoch")
plt.ylabel("MAE (scaled)")
plt.legend()
plt.show()

## 5.8 Evaluation on Test Set (Scaled)

The best GRU model was evaluated on the test dataset using the scaled target values produced during preprocessing. The predictions were compared with the ground truth values in the scaled space, and several evaluation metrics were computed to assess model performance.

The model achieved low error values and a high coefficient of determination (R²), indicating that it captures most of the variance in the temperature series and produces accurate multi-step forecasts.

Since the values are still in the scaled space, the Mean Absolute Percentage Error (MAPE) was not computed at this stage, as percentage-based metrics are not meaningful before transforming the predictions back to the original temperature units.

The evaluation results are summarised in Table X.

| Metric | Value (scaled) |
|------|------|
| MAE | 0.1885 |
| RMSE | 0.2493 |
| R² | 0.9239 |

**Table 19** - Final Model Performance Metrics

In [55]:
best_gru = best["model"]
y_pred_gru_scaled = best_gru.predict(X_test, verbose=0)

yt = y_test.reshape(-1)
yp = y_pred_gru_scaled.reshape(-1)

mae = mean_absolute_error(yt, yp)
rmse = np.sqrt(mean_squared_error(yt, yp))
r2 = r2_score(yt, yp)

# MAPE on scaled units is not meaningful; we'll compute MAPE in °C after inverse-scaling.
print("TEST (scaled) - MAE:", mae, "RMSE:", rmse, "R2:", r2)

## 5.9 Inverse Scaling and Metrics in °C

To obtain interpretable evaluation metrics, the predicted and ground-truth temperature values were transformed back to the original temperature scale (degrees Celsius). This was done by applying the inverse of the standardisation used during preprocessing.

Since the data were standardised using `StandardScaler`, the inverse transformation was performed using the mean and standard deviation associated with the temperature feature. Only the target variable was inverse-scaled, while the remaining input features remained in the scaled space.

After converting the predictions and targets back to degrees Celsius, evaluation metrics were recomputed to assess the forecasting accuracy in physically meaningful units.

While the scaled metrics provide useful information about optimisation behaviour during training, evaluating the model in degrees Celsius allows a clearer interpretation of the forecasting error.

In [56]:
def inverse_scale_target(y_scaled, scaler, feature_names, target_col):
    t_idx = feature_names.index(target_col)
    mean = scaler.mean_[t_idx]
    std  = scaler.scale_[t_idx]
    return y_scaled * std + mean

y_test_c = inverse_scale_target(y_test, scaler, feat_cols, target_col)
y_pred_c = inverse_scale_target(y_pred_gru_scaled, scaler, feat_cols, target_col)

### 5.9.1 Metrics in °C

After applying the inverse transformation, evaluation metrics were computed in degrees Celsius to obtain physically interpretable results.

The GRU model achieves a Mean Absolute Error (MAE) of 1.63 °C and a Root Mean Squared Error (RMSE) of 2.15 °C on the test set. These values indicate that, on average, the model’s predictions deviate by roughly one to two degrees Celsius from the observed temperature.

The coefficient of determination (R² = 0.92) shows that the model explains more than 92% of the variance in the temperature series, indicating that the dominant temporal patterns are successfully captured.

The Mean Absolute Percentage Error (MAPE) was not used as an evaluation metric. Temperature values in the dataset frequently approach or cross zero degrees Celsius, which makes percentage-based errors unstable due to division by values close to zero. In such situations, MAPE can produce extremely large or misleading values. For this reason, MAE and RMSE are considered more reliable metrics for evaluating temperature forecasting models.

In [57]:
eps = 1e-6
yt = y_test_c.reshape(-1)
yp = y_pred_c.reshape(-1)

mae_c = mean_absolute_error(yt, yp)
rmse_c = np.sqrt(mean_squared_error(yt, yp))
r2_c = r2_score(yt, yp)

print("TEST (°C) - MAE:", mae_c, "RMSE:", rmse_c, "R2:", r2_c)

## 5.10 MAE by Horizon

To analyse how forecast accuracy evolves with increasing prediction horizon, the Mean Absolute Error (MAE) was computed separately for each of the 24 forecast hours.

The results show a gradual increase in error as the forecasting horizon expands. The MAE starts at approximately 0.6 °C for the first hour ahead* reflecting very accurate short-term predictions. As the horizon increases, the error progressively grows, reaching around 2.1–2.2 °C at the 24-hour horizon.

This behaviour is expected in multi-step forecasting tasks. When predictions are made further into the future, the model must rely on increasingly uncertain temporal patterns, which naturally leads to larger prediction errors.

Despite this gradual increase, the error growth remains smooth and stable, indicating that the model maintains reliable predictive performance across the full 24-hour forecasting window. The absence of sudden jumps in error suggests that the GRU architecture successfully captures the temporal dynamics required for medium-term temperature forecasting.

This result confirms that the model remains robust even for longer forecasting horizons, which is particularly important for practical meteorological applications where predictions for the next 24 hours are required.

In [58]:
H = y_test.shape[1]
mae_by_h = np.mean(np.abs(y_test_c - y_pred_c), axis=0)

plt.figure()
plt.plot(range(1, H+1), mae_by_h)
plt.title("Best GRU - MAE by Horizon (°C)")
plt.xlabel("Horizon (hours ahead)")
plt.ylabel("MAE (°C)")
plt.show()

**Figure 72** - Best GRU: MAE by Forecast Horizon (°C)


In [59]:
results_df.to_csv("gru_random_search_results.csv", index=False)

# 6. Overview of Hyperparameter Configurations

This section summarises the hyperparameter configurations explored during the optimisation process for the GRU forecasting model. Each configuration represents a specific combination of architectural parameters, regularisation strategies and optimisation settings sampled during the Optuna search procedure.

For each trial, the table reports the validation loss and the corresponding validation MAE obtained during training. These metrics allow direct comparison between different configurations and help identify which design choices lead to better predictive performance.

The results are sorted by validation loss, where the top rows correspond to the most accurate models and the bottom rows represent the least effective configurations. In total, 150 trials were executed during the optimisation process, of which 92 completed successfully while 58 were pruned early by the Hyperband strategy.

The best-performing configuration achieved a validation loss of 0.0337 and a validation MAE of 0.1924, demonstrating strong predictive performance for the multi-horizon temperature forecasting task.

Examining the top-performing models reveals a consistent pattern in the hyperparameter choices. Most of the best trials share the following characteristics:

* Two GRU layers, suggesting that moderate architectural depth is sufficient to capture temporal dependencies.
* Hidden sizes around 96–64 units, indicating that medium-capacity models perform better than either very small or excessively large networks.
* No dropout, implying that strong regularisation through dropout was not necessary given the size of the dataset.
* AdamW optimiser with small weight decay, which appears to provide stable optimisation behaviour.
* Low learning rates (around 2×10⁻⁴), allowing smoother convergence during training.
* Huber loss, which is robust to occasional large prediction errors.

In contrast, the worst-performing configurations often include one or more of the following characteristics:

* High dropout values (0.3–0.4), which may excessively regularise the model and limit its ability to learn meaningful temporal patterns.
* Large Gaussian noise levels, which can destabilise training.
* Higher learning rates, which may lead to less stable optimisation.
* Different loss functions such as MAE or MSE, which in some trials produced less stable convergence compared to Huber loss.

The comparison between the best and worst trials highlights the importance of balanced architectural complexity and stable optimisation settings when training recurrent neural networks for time-series forecasting.

In [60]:
display(results_df.head(10))     # top 10
display(results_df.tail(10))     # worst 10
print("Configs tested:", len(results_df))
print("Best val_loss:", results_df["best_val_loss"].min())
print("Best val_mae:", results_df["best_val_mae"].min())

## 6.1 Distribution of Validation Performance

To better understand the variability of model performance across the explored configurations, histograms were generated for the best validation loss and best validation MAE obtained during training.

The distribution of validation loss shows a strong concentration of trials in the lower error region, particularly between 0.033 and 0.040 MSE. This indicates that many hyperparameter configurations produce reasonably accurate models once the optimisation process converges.

However, a small number of configurations result in significantly higher errors, with validation loss values extending beyond 0.07 and up to approximately 0.21. These outliers correspond to less effective combinations of hyperparameters, such as high dropout values, unstable optimisation settings, or insufficient model capacity.

A similar pattern can be observed in the distribution of validation MAE. Most trials cluster around 0.19–0.20, suggesting that a large portion of the explored configurations achieve comparable predictive performance. A few configurations, however, produce noticeably larger errors, indicating weaker generalisation.

These distributions highlight two important aspects of the optimisation process. First, the search space contains multiple viable configurations capable of achieving strong forecasting performance. Second, certain hyperparameter combinations clearly degrade model quality, reinforcing the value of systematic hyperparameter optimisation.

The presence of relatively few high-error trials also reflects the effectiveness of the Hyperband pruning strategy, which terminated many underperforming configurations early and prevented unnecessary training of poor models.

The narrow spread among the best-performing trials suggests that the optimisation process converged to a stable region of the hyperparameter space where several configurations yield very similar performance.

In [61]:
plt.figure()
plt.hist(results_df["best_val_loss"], bins=20)
plt.title("Distribution of best validation loss (MSE)")
plt.xlabel("best_val_loss")
plt.ylabel("count")
plt.show()

plt.figure()
plt.hist(results_df["best_val_mae"], bins=20)
plt.title("Distribution of best validation MAE")
plt.xlabel("best_val_mae")
plt.ylabel("count")
plt.show()

## 6.2 Hyperparameter Search Progress and Learning Rate Analysis

The hyperparameter optimisation progress is illustrated in Figure X. Each point represents the best validation loss obtained from a completed trial, while the red line shows the running best value observed during the optimisation process. The vertical dashed line indicates the trial where the best-performing configuration was discovered.

The results show that a strong configuration was identified relatively early in the optimisation process. After this point, most additional trials produced validation losses very close to the current best value, indicating that the search quickly converged to a stable region of the hyperparameter space.

The distribution of validation loss across different learning rates provides additional insight into the sensitivity of the model to this parameter. The boxplot analysis shows that learning rates around 0.0002 consistently produce the lowest validation errors and the most stable training behaviour.

In contrast, higher learning rates such as 0.0003 display much larger variability in validation performance, suggesting that the optimisation process becomes less stable at these values. Very small or very large learning rates also tend to produce slightly worse results.

Overall, this analysis confirms that the learning rate is one of the most influential hyperparameters in training the GRU model. A moderately small learning rate enables stable gradient updates and allows the model to converge to a better-performing solution.

In [62]:
# Search progress chart
# results_df is sorted by val_loss; we recover search order via the config names (gru_rs_01...gru_rs_60)
results_order = results_df.sort_values("name").reset_index(drop=True)
x = range(1, len(results_order) + 1)
running_best = results_order["best_val_loss"].cummin()
best_idx = int(results_order["best_val_loss"].idxmin()) + 1  # 1-indexed position

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: val_loss per config + running best
ax = axes[0]
ax.scatter(x, results_order["best_val_loss"], alpha=0.5, s=30, label="Config val_loss")
ax.plot(x, running_best, color="red", linewidth=2, label="Running best")
ax.axvline(best_idx, color="red", linestyle="--", alpha=0.4, label=f"Best found at #{best_idx}")
ax.set_title("Random Search Progress")
ax.set_xlabel("Config index (search order)")
ax.set_ylabel("Best val_loss")
ax.legend()

# Right: val_loss by learning rate
ax = axes[1]
results_df.boxplot(column="best_val_loss", by="lr", ax=ax)
ax.set_title("Val loss by learning rate")
plt.suptitle("")
ax.set_xlabel("Learning rate")
ax.set_ylabel("best_val_loss")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**Figure 75** - Random Search Progress and Validation Loss by Learning Rate

## 6.3 Performance vs Training Time

The scatter plot illustrates the relationship between training time and validation performance for the different hyperparameter configurations evaluated during the optimisation process.

Each point represents a single configuration. The horizontal axis corresponds to the total training time (in seconds), while the vertical axis shows the best validation loss obtained by that model. Since lower validation loss indicates better predictive performance, points located closer to the bottom of the plot correspond to more accurate models.

The results reveal that longer training times do not necessarily translate into better model performance. Many of the best-performing configurations are concentrated within a moderate training-time range of approximately 120–180 seconds, where several models achieve validation losses close to the optimal value.

In contrast, some configurations with very short training times produce higher validation losses, indicating insufficient model capacity or unstable optimisation. At the same time, a few models with longer training times do not provide additional performance improvements, suggesting diminishing returns from increased computational cost.

This analysis highlights the importance of balancing model complexity and computational efficiency. Rather than simply increasing training time or model size, carefully tuned configurations can achieve strong predictive performance while maintaining reasonable computational cost. Such configurations represent favourable trade-offs between accuracy and efficiency.

This type of analysis provides a Pareto-style interpretation of the optimisation results, helping identify configurations that offer strong predictive accuracy while maintaining efficient training times.

In [63]:
plt.figure()
plt.scatter(results_df["train_time_sec"], results_df["best_val_loss"], alpha=0.7)
plt.title("Validation loss vs training time")
plt.xlabel("train_time_sec")
plt.ylabel("best_val_loss (lower is better)")
plt.show()

**Figure 76** - Validation Loss vs Training Time


## 6.4 Important Hyperparameters

To better understand how different architectural choices influence model performance, additional analyses were conducted focusing on key hyperparameters of the GRU architecture. In particular, we examine how validation performance varies with respect to the number of GRU layers, dropout rate, and optimizer choice.

These analyses provide insight into which hyperparameters have the strongest impact on forecasting performance and help guide the selection of more effective model configurations. By studying the distribution of validation loss across different hyperparameter values, we can identify patterns that reveal which architectural or training choices tend to produce more stable and accurate models.

### 6.4.1 Effect of the Number of GRU Layers

The boxplot above illustrates the distribution of validation loss across models with different numbers of GRU layers.

Models with two GRU layers consistently achieve the lowest validation losses, including the best-performing configurations identified during the optimisation process. This suggests that a moderate level of architectural depth helps the model capture more complex temporal dependencies present in the meteorological time series.

Models with a single GRU layer also achieve relatively low validation losses, although their performance is slightly less consistent compared to the two-layer architecture. This indicates that shallow architectures are still capable of modelling the underlying temporal structure, but may lack the representational capacity of deeper models.

In contrast, configurations with three GRU layers exhibit substantially higher validation losses. This suggests that deeper architectures introduce unnecessary complexity for this task, potentially making optimisation more difficult and increasing the risk of unstable training.

The results indicate that moderately deep architectures provide the best trade-off between representational capacity and training stability, with two-layer GRU models offering the most reliable forecasting performance.

This behaviour suggests the existence of a capacity “sweet spot”, where the model is sufficiently expressive to capture temporal patterns without introducing unnecessary complexity.

In [64]:
plt.figure()
results_df.boxplot(column="best_val_loss", by="n_layers")
plt.title("Val loss by number of GRU layers")
plt.suptitle("")
plt.xlabel("n_layers")
plt.ylabel("best_val_loss")
plt.show()

**Figure 77** - Validation Loss by Number of GRU Layers

### 6.4.2 Effect of Dropout Regularization

The boxplot illustrates how validation performance varies across different dropout rates used during training.

The results show that configurations without dropout (0.0) consistently achieve the lowest validation losses. These models not only reach the best overall performance but also display very low variability across configurations, indicating stable and reliable training behaviour.

As the dropout rate increases, the validation loss generally becomes larger and more variable. In particular, models with dropout values of 0.1 and above exhibit substantially higher validation losses and greater dispersion, suggesting that stronger regularisation negatively affects the model’s ability to learn useful temporal representations.

Higher dropout values remove a larger fraction of the hidden activations during training, which can significantly reduce the effective capacity of the recurrent network. In this forecasting task, where the dataset is relatively large and the temporal structure is complex, excessive dropout appears to hinder the learning process.

Overall, the results indicate that little or no dropout regularisation is preferable for this task, allowing the GRU architecture to fully exploit the temporal information present in the meteorological time series.

This observation is consistent with the behaviour of the best-performing configuration, which also used no dropout regularisation.

In [65]:
plt.figure()
results_df.boxplot(column="best_val_loss", by="dropout")
plt.title("Val loss by dropout")
plt.suptitle("")
plt.xlabel("dropout")
plt.ylabel("best_val_loss")
plt.show()

**Figure 78** - Validation Loss by Dropout Rate

### 6.4.3 Effect of the Optimizer

The final analysis compares model performance across the different optimization algorithms used during training.

The boxplot shows that configurations trained with AdamW consistently achieve lower validation losses and exhibit much lower variability compared to those using the standard Adam optimizer. Most of the best-performing models are clustered near the lowest validation loss values when AdamW is used.

In contrast, models trained with Adam display substantially greater dispersion in validation loss, with several configurations producing significantly worse results. This indicates that the optimisation process is less stable when using the standard Adam optimizer for this task.

A likely explanation is that AdamW applies decoupled weight decay, which improves regularization by separating weight decay from the gradient-based update rule. This mechanism often leads to more stable optimisation and better generalisation performance in deep learning models.

The results suggest that AdamW provides a more reliable and effective optimisation strategy for training the GRU-based forecasting model on the Jena climate dataset.

This observation is consistent with the best-performing configuration identified during the hyperparameter optimisation process, which also used the AdamW optimizer.

In [66]:
plt.figure()
results_df.boxplot(column="best_val_loss", by="optimizer")
plt.title("Val loss by optimizer")
plt.suptitle("")
plt.xlabel("optimizer")
plt.ylabel("best_val_loss")
plt.show()

**Figure 79** - Validation Loss by Optimizer Type

# 7. Model Evaluation and Forecast Analysis

This section presents a detailed evaluation of the forecasting performance of the proposed GRU model on the test dataset. Beyond aggregate performance metrics, the analysis focuses on how the model behaves across different prediction horizons and how effectively it captures the temporal dynamics of the temperature series.

Several complementary analyses are used to assess the robustness and reliability of the model. These include a comparison with a simple baseline model, an examination of forecast errors across the 24-hour horizon, and visual inspection of predicted and observed temperature trajectories.

These analyses provide a more complete understanding of the strengths and limitations of the GRU architecture for multi-step temperature forecasting.

## 7.1 Horizon-wise Comparison: Baseline vs Best GRU

To evaluate the effectiveness of the GRU forecasting model, its performance was compared against a persistence baseline across all 24 forecast horizons. The persistence model assumes that future temperature remains equal to the most recent observed value, making it a simple but relevant benchmark for short-term forecasting.

Figure X shows the Mean Absolute Error (MAE) in degrees Celsius for each horizon from 1 to 24 hours ahead. At the shortest horizon, both models achieve similar performance, which is expected since very near-future temperatures are often close to the most recent observation.

However, as the forecasting horizon increases, the persistence baseline deteriorates rapidly, with the MAE rising sharply and remaining substantially higher than that of the GRU model. This behaviour reflects the inability of the naive baseline to capture the temporal evolution of temperature across the day.

In contrast, the GRU model maintains consistently lower error across the full forecasting horizon. Although its error also increases gradually with horizon, the increase is smooth and much more controlled. This indicates that the GRU successfully learns meaningful temporal dependencies from the historical meteorological observations and is able to generate more reliable multi-step forecasts.

The comparison demonstrates that the GRU architecture provides a substantial improvement over a naive persistence strategy, particularly for medium- and longer-range horizons, where modelling temporal dynamics becomes essential.

The increasing performance gap between the two models as the horizon expands highlights the added value of learned temporal representations for multi-step forecasting.

In [67]:
# Convert persistence baseline to °C
y_pred_persist_c = inverse_scale_target(y_pred_persist, scaler, feat_cols, target_col)

H = y_test.shape[1]
mae_h_p = np.mean(np.abs(y_test_c - y_pred_persist_c), axis=0)
mae_h_g = np.mean(np.abs(y_test_c - y_pred_c), axis=0)

plt.figure()
plt.plot(range(1, H+1), mae_h_p, label="Persistence")
plt.plot(range(1, H+1), mae_h_g, label="Best GRU")
plt.title("MAE by horizon (°C) - Baseline vs GRU")
plt.xlabel("Horizon (hours ahead)")
plt.ylabel("MAE (°C)")
plt.legend()
plt.show()

**Figure 80** - Horizon-wise MAE Comparison: Baseline vs Best GRU

## 7.2 Error Distribution (Residual Analysis)

To further analyse the forecasting performance of the GRU model, the distribution of prediction residuals on the test set was examined. Residuals are defined as the difference between the true temperature values and the predicted values produced by the model.

The residual histogram is approximately centred around zero, indicating that the model does not exhibit strong systematic bias in its predictions. This suggests that the GRU model is not consistently overestimating or underestimating temperature values across the dataset.

The shape of the distribution is roughly symmetric and resembles a bell-shaped curve, which is typical of well-behaved regression residuals. Most residuals are concentrated close to zero, indicating that the majority of predictions deviate only slightly from the true temperature values.

The absolute error distribution provides additional insight into the magnitude of prediction errors. The histogram shows that most absolute errors are relatively small, with a large proportion of predictions falling within a few degrees Celsius of the true value. Larger errors occur less frequently and appear in the tails of the distribution.

Overall, the residual analysis confirms that the GRU model produces stable and generally accurate predictions, with most errors remaining within a reasonable range for short- to medium-term temperature forecasting.

Most absolute prediction errors are concentrated below approximately 3 °C, with only a small number of larger deviations.

In [68]:
res = (y_test_c - y_pred_c).reshape(-1)

plt.figure()
plt.hist(res, bins=60)
plt.title("Residual distribution (True - Pred) in °C")
plt.xlabel("Residual (°C)")
plt.ylabel("Count")
plt.show()

# Absolute error distribution
abs_err = np.abs(res)
plt.figure()
plt.hist(abs_err, bins=60)
plt.title("Absolute error distribution (°C)")
plt.xlabel("|Error| (°C)")
plt.ylabel("Count")
plt.show()

## 7.3 One-Step-Ahead Forecast Visualization

To visually assess the quality of the GRU forecasts, a one-week slice of the test set was plotted, comparing the true temperature values with the corresponding 1-hour-ahead predictions.

The figure shows that the predicted series closely follows the temporal dynamics of the observed signal. The model successfully captures the main oscillatory behaviour of the temperature series, including most local increases, decreases, peaks, and troughs.

Although small deviations can be observed at some turning points, the overall agreement between predicted and true values remains strong. In particular, the model tends to slightly smooth some of the sharpest fluctuations, which is a common behaviour in neural forecasting models.

This visual comparison confirms that the GRU model provides accurate short-term forecasts and is able to track the local evolution of temperature with high fidelity.

The close alignment between the two curves supports the quantitative results previously obtained for the short forecasting horizons.

In [69]:
start = 0
n = 7 * 24  # 1 week of windows

y_true_1 = y_test_c[start:start+n, 0]
y_pred_1 = y_pred_c[start:start+n, 0]

plt.figure(figsize=(10,4))
plt.plot(y_true_1, label="True (h=1)")
plt.plot(y_pred_1, label="Pred (h=1)")
plt.title("1-hour ahead forecast (1-week slice, °C)")
plt.xlabel("Test window index")
plt.ylabel("T (degC)")
plt.legend()
plt.show()

**Figure 83** - One-Step-Ahead Forecast: True vs Predicted Temperatures

## 7.4 Multi-Step Forecast Trajectories

To better understand the model's behaviour across longer prediction horizons, several 24-hour forecast trajectories were visualised for different examples in the test set.

Each plot compares the true temperature evolution with the sequence of predictions produced by the GRU model for the full 24-hour forecasting horizon. These examples illustrate how the model performs when predicting multiple future time steps simultaneously.

Across the different cases, the predicted trajectories generally follow the overall trends of the true temperature curves. The model successfully captures the broad evolution of temperature, including rising and falling patterns over the forecast horizon.

However, deviations between predicted and true values become more noticeable as the horizon increases. In some examples, the model slightly underestimates temperature peaks or smooths sharper transitions in the true signal. This behaviour is common in multi-step forecasting tasks, where prediction uncertainty accumulates as the forecast extends further into the future.

Despite these deviations, the GRU model consistently reproduces the main temporal dynamics of the temperature evolution, indicating that it has learned meaningful patterns from the historical meteorological data.

These visual results are consistent with the quantitative evaluation presented earlier, where prediction error increases gradually with the forecast horizon but remains within a reasonable range.

In [70]:
examples = [50, 200, 500]  # pick any indices within test

for i in examples:
    plt.figure(figsize=(10,4))
    plt.plot(range(1, H+1), y_test_c[i], label="True")
    plt.plot(range(1, H+1), y_pred_c[i], label="Pred")
    plt.title(f"24-hour forecast trajectory example (test idx={i})")
    plt.xlabel("Horizon (hours ahead)")
    plt.ylabel("T (degC)")
    plt.legend()
    plt.show()

## 7.5 Learning Curves of the Best Configurations

To analyse the training dynamics of the most promising configurations, the learning curves of the top-performing models were examined. These curves show the evolution of both training loss and validation loss over the training epochs.

Across all three configurations, the training loss decreases rapidly during the first few epochs, indicating that the model quickly learns the main temporal patterns present in the data. After this initial phase, the rate of improvement slows and the curves gradually converge.

The validation loss follows a similar trend and stabilizes after approximately 12–15 epochs, suggesting that the model reaches its optimal performance relatively early during training. Beyond this point, additional epochs provide only marginal improvements.

A small gap between training and validation loss can be observed toward the later epochs. This behaviour is typical in neural network training and indicates mild overfitting, but the gap remains relatively small, suggesting that the model still generalizes well to unseen data.

Learning curves demonstrate stable and well-behaved training dynamics for the selected GRU configurations, confirming that the chosen hyperparameters enable efficient optimisation without severe overfitting.

The similarity between the curves of the top configurations also indicates that the optimisation process converged to a stable region of the hyperparameter space.

In [71]:
# Retrain top-3 configs to visualise their learning curves.
# config_keys comes from the Optuna study so all sampled parameters are included.
config_keys = list(study.best_params.keys())

top3 = results_df.head(3).to_dict(orient="records")
histories = []

for cfg_row in top3:
    cfg = {k: cfg_row[k] for k in config_keys if k in cfg_row}
    model, history, best_val_loss, best_val_mae = train_one_config(
        cfg, X_train, y_train, X_val, y_val, max_epochs=40
    )
    histories.append((cfg_row["name"], history.history))

for name, h in histories:
    plt.figure()
    plt.plot(h["val_loss"],  label="val_loss")
    plt.plot(h["loss"],      label="train_loss")
    plt.title(f"Loss curves - {name}")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.show()


# 8. Forecasting

Beyond quantitative evaluation, an important objective of time-series forecasting models is their ability to generate future predictions from the most recent available observations. In practical settings, forecasting systems are applied to the latest measured data in order to estimate the future evolution of the target variable.

This section illustrates how the trained GRU model can be used for inference. Using the last available observation window from the dataset, the model generates a direct 24-hour temperature forecast, representing the expected evolution of air temperature over the next day.

## 8.1 Generating a 24-Hour Temperature Forecast

To generate a future forecast, the final input window of length L = 120 hours was extracted from the most recent observations in the cleaned hourly dataset. The same feature engineering steps used during training were applied, including the cyclical temporal covariates and wind direction encoding.

The resulting input window was then transformed using the previously fitted scaler, ensuring consistency between training and inference. The scaled input was provided to the trained GRU model, which produced a sequence of 24 temperature predictions corresponding to the next 24 hours.

Because the model outputs predictions in the scaled space, the forecast was transformed back to degrees Celsius using the inverse of the target scaling transformation. The resulting values were then associated with their corresponding future timestamps.

The forecasted trajectory shows a plausible daily temperature cycle, with colder values during the early hours, a gradual increase toward the afternoon, and a subsequent decline during the evening. This behaviour is consistent with the daily periodic structure previously identified during the exploratory data analysis.

It should be noted that this forecast extends beyond the time span covered by the dataset. Therefore, it is presented as an illustration of the model’s inference capability rather than as an additional quantitative evaluation.

| Date Time           | T_forecast_degC |
|---------------------|-----------------|
| 2017-01-01 01:00:00 | -4.501 |
| 2017-01-01 02:00:00 | -4.438 |
| 2017-01-01 03:00:00 | -4.051 |
| 2017-01-01 04:00:00 | -3.865 |
| 2017-01-01 05:00:00 | -4.124 |
| 2017-01-01 06:00:00 | -4.282 |
| 2017-01-01 07:00:00 | -3.877 |
| 2017-01-01 08:00:00 | -3.384 |
| 2017-01-01 09:00:00 | -2.765 |
| 2017-01-01 10:00:00 | -1.891 |
| 2017-01-01 11:00:00 | -1.095 |
| 2017-01-01 12:00:00 | -0.569 |
| 2017-01-01 13:00:00 | 0.835 |
| 2017-01-01 14:00:00 | 1.454 |
| 2017-01-01 15:00:00 | 1.412 |
| 2017-01-01 16:00:00 | 1.334 |
| 2017-01-01 17:00:00 | 0.787 |
| 2017-01-01 18:00:00 | 0.056 |
| 2017-01-01 19:00:00 | 0.091 |
| 2017-01-01 20:00:00 | -0.870 |
| 2017-01-01 21:00:00 | -0.943 |
| 2017-01-01 22:00:00 | -0.787 |
| 2017-01-01 23:00:00 | -1.543 |
| 2017-01-02 00:00:00 | -1.578 |

**Table 20** - Forecast Results for 24 Hours

In [72]:
# 2.1) Build last input window (L hours) from the end of the dataset
df_full = df_work.copy()  # df_work: hourly clean, index=Date Time, columns=features (unscaled)

# If you engineered features during training, apply the same here:
df_full_fe = add_time_features(df_full)

# Keep the same columns and order used in scaling/training
df_full_fe = df_full_fe[feat_cols].copy()

# Take last L rows
last_window = df_full_fe.iloc[-L:].copy()

# Scale
last_window_scaled = scaler.transform(last_window.values)

# Model input shape: (1, L, n_features)
X_last = last_window_scaled.reshape(1, L, len(feat_cols))

# Predict next 24h (scaled)
y_next24_scaled = best_gru.predict(X_last, verbose=0)  # shape (1, 24)

# Inverse scale to °C
y_next24_c = inverse_scale_target(y_next24_scaled, scaler, feat_cols, target_col).flatten()

# Build future datetime index
last_time = df_full_fe.index[-1]
future_index_24 = pd.date_range(start=last_time + pd.Timedelta(hours=1), periods=H, freq="1h")

# Plot forecast
plt.figure(figsize=(10,4))
plt.plot(future_index_24, y_next24_c, marker="o")
plt.title("Forecast - Next 24 hours (Temperature)")
plt.xlabel("Date Time")
plt.ylabel("T (degC)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# Show as table
forecast_24h = pd.DataFrame({"Date Time": future_index_24, "T_forecast_degC": y_next24_c})
display(forecast_24h.head(10))
display(forecast_24h.tail(10))

**Figure 90** - 24-Hour Temperature Forecast 

## 8.2 Five-Day Forecasting

To further demonstrate the practical forecasting capability of the trained model, a longer prediction horizon is considered by generating a five-day temperature forecast (120 hours). Since the GRU model is trained to predict the next 24 hours, a rolling forecasting strategy is used to iteratively extend the prediction horizon.

At each step, the model receives the most recent input window and predicts the next 24 hours of temperature. The predicted values are then appended to the simulated dataset and incorporated into the input window used for the following prediction step. This recursive procedure is repeated until the full five-day horizon is obtained.

For the remaining meteorological variables used as inputs (such as pressure, humidity and wind-related measurements), future values are approximated using observations from a previous time lag. This approach leverages the quasi-periodic behaviour commonly observed in meteorological data and provides a reasonable approximation of unknown future conditions.

The resulting forecast represents a continuous multi-day temperature trajectory generated by the GRU model. The predicted sequence exhibits a clear daily cycle, with alternating temperature peaks and troughs that reflect the diurnal pattern previously observed in the dataset.

It is important to note that uncertainty increases as the forecasting horizon grows. Because the model recursively uses its own predictions as inputs, small prediction errors can accumulate over time, potentially leading to larger deviations in longer horizons. Despite this limitation, the model is still able to reproduce the general structure of the temperature dynamics over several consecutive days.

Figure X shows the five-day temperature forecast generated using the rolling prediction strategy.

In [73]:
def forecast_rolling_5days(df_full_raw, best_model, scaler, feat_cols, target_col, L=120, H=24, days=5, lag_hours=168):
    steps = days * 24  # total future hours
    n_blocks = steps // H

    df_sim = df_full_raw.copy()
    preds_all = []
    last_time = df_sim.index[-1]

    for b in range(n_blocks):
        # Recompute time features over current (growing) df_sim
        df_sim_fe = add_time_features(df_sim)

        # Use last L hours as model input
        X_win = df_sim_fe[feat_cols].iloc[-L:].values
        X_win_s = scaler.transform(X_win).reshape(1, L, len(feat_cols))

        y_block_s = best_model.predict(X_win_s, verbose=0)  # (1, H)
        y_block_c = inverse_scale_target(y_block_s, scaler, feat_cols, target_col).flatten()

        # Future timestamps for this block
        start = last_time + pd.Timedelta(hours=1)
        idx = pd.date_range(start=start, periods=H, freq="1h")

        preds_all.append(pd.DataFrame({"Date Time": idx, "T_forecast_degC": y_block_c}))

        # Seasonal-naive imputation for exogenous features:
        # For each future hour t, use the observed values from lag_hours ago (default: 168h = 7 days).
        # This is physically motivated: atmospheric variables (pressure, humidity, wind)
        # exhibit strong weekly periodicity, making same-hour-last-week a better
        # proxy than freezing the last observed value indefinitely.
        for t, yv in zip(idx, y_block_c):
            lag_time = t - pd.Timedelta(hours=lag_hours)
            if lag_time in df_sim.index:
                new_row = df_sim.loc[[lag_time]].copy()
            else:
                new_row = df_sim.iloc[-1:].copy()  # fallback if lag not in history
            new_row.index = [t]
            new_row[target_col] = yv  # temperature always from model prediction
            df_sim = pd.concat([df_sim, new_row])

        last_time = idx[-1]

    return pd.concat(preds_all, ignore_index=True)

forecast_5d = forecast_rolling_5days(df_work, best_gru, scaler, feat_cols, target_col, L=L, H=H, days=5)
display(forecast_5d.head())
display(forecast_5d.tail())

### 8.2.1 Five-Day Temperature Forecast

To extend the forecasting horizon beyond the initial 24-hour prediction window, a rolling forecasting strategy is applied to generate a five-day temperature forecast. The model iteratively predicts the next 24 hours and then incorporates these predictions into the input sequence to produce subsequent forecasts.

The resulting predictions are combined to form a continuous temperature trajectory covering the next 120 hours (5 days). This approach allows the model to simulate longer-term forecasts while maintaining the same temporal structure used during training.

The figure below illustrates the predicted evolution of temperature over the five-day horizon. The forecast exhibits smooth temporal patterns that resemble realistic daily temperature cycles, indicating that the GRU model captures meaningful temporal dynamics in the data.

In [74]:
plt.figure(figsize=(10,4))
plt.plot(forecast_5d["Date Time"], forecast_5d["T_forecast_degC"])
plt.title("Forecast - Next 5 days (Temperature) [rolling]")
plt.xlabel("Date Time")
plt.ylabel("T (degC)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

**Figure 91** - Five-Day Temperature Forecast